# Phase 3 — NOAA Weather Integration

This notebook integrates hourly NOAA weather observations with the 2025 BTS flight operations dataset.

The weather source is NOAA's Global Historical Climatology Network Hourly (GHCNh), which provides hourly and synoptic land-based weather observations.

The integration pipeline will follow:

BTS Airport
→ Airport Coordinates
→ NOAA Weather Station
→ Hourly Weather Observation
→ Leakage-Safe Weather Features
→ Flight Record

The objective is to determine whether weather information provides predictive value beyond the BTS-only Baseline v2 model.

December 2025 remains untouched as the final test period.

In [1]:
from pathlib import Path

import duckdb
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent

INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
RAW_WEATHER_DIR = PROJECT_ROOT / "data" / "raw" / "weather"

con = duckdb.connect()

RAW_WEATHER_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
all_flights_path = INTERIM_DIR / "flights_2025_*.parquet"

con.execute(
    f"""
    CREATE OR REPLACE VIEW flights_2025 AS
    SELECT *
    FROM read_parquet('{all_flights_path}')
    """
)

In [3]:
con.sql("""
SELECT
    COUNT(*) AS rows,
    MIN(FlightDate) AS first_date,
    MAX(FlightDate) AS last_date
FROM flights_2025
""").df()

,rows,first_date,last_date
0,7001619,2025-01-01,2025-12-31


In [4]:
con.sql("""
DESCRIBE flights_2025
""").df()

,column_name,column_type,null,key,default,extra
0,Year,BIGINT,YES,None,None,None
1,Quarter,BIGINT,YES,None,None,None
2,Month,BIGINT,YES,None,None,None
3,DayofMonth,BIGINT,YES,None,None,None
4,DayOfWeek,BIGINT,YES,None,None,None
...,...,...,...,...,...,...
105,Div5TotalGTime,VARCHAR,YES,None,None,None
106,Div5LongestGTime,VARCHAR,YES,None,None,None
107,Div5WheelsOff,VARCHAR,YES,None,None,None
108,Div5TailNum,VARCHAR,YES,None,None,None


In [5]:
columns = con.sql("""
DESCRIBE flights_2025
""").df()

columns[
    columns["column_name"].str.contains(
        "lat|lon|airport|origin|dest",
        case=False,
        regex=True
    )
]

,column_name,column_type,null,key,default,extra
11,OriginAirportID,BIGINT,YES,None,None,None
12,OriginAirportSeqID,BIGINT,YES,None,None,None
13,OriginCityMarketID,BIGINT,YES,None,None,None
14,Origin,VARCHAR,YES,None,None,None
15,OriginCityName,VARCHAR,YES,None,None,None
16,OriginState,VARCHAR,YES,None,None,None
17,OriginStateFips,VARCHAR,YES,None,None,None
18,OriginStateName,VARCHAR,YES,None,None,None
19,OriginWac,BIGINT,YES,None,None,None
20,DestAirportID,BIGINT,YES,None,None,None


In [6]:
airport_master = con.sql("""
SELECT DISTINCT
    Origin AS airport,
    OriginAirportID AS bts_airport_id,
    OriginCityName AS city,
    OriginState AS state
FROM flights_2025

WHERE Origin IS NOT NULL

ORDER BY Origin
""").df()

print("Airports:", len(airport_master))

airport_master.head(10)

Airports: 352


,airport,bts_airport_id,city,state
0,ABE,10135,"Allentown/Bethlehem/Easton, PA",PA
1,ABI,10136,"Abilene, TX",TX
2,ABQ,10140,"Albuquerque, NM",NM
3,ABR,10141,"Aberdeen, SD",SD
4,ABY,10146,"Albany, GA",GA
5,ACK,10154,"Nantucket, MA",MA
6,ACT,10155,"Waco, TX",TX
7,ACV,10157,"Arcata/Eureka, CA",CA
8,ACY,10158,"Atlantic City, NJ",NJ
9,ADK,10165,"Adak Island, AK",AK


In [7]:
airport_master.groupby("airport").size().value_counts()

1    352
Name: count, dtype: int64

## Airport Geospatial Reference Data

BTS identifies airports using airport codes and internal airport identifiers but does not provide latitude and longitude coordinates in the flight-performance dataset.

Airport coordinates are required to associate each BTS airport with an appropriate NOAA weather station.

FAA aeronautical airport data will therefore be used as the authoritative geospatial reference source. The airport reference-point coordinates will support geographic matching between BTS airports and NOAA weather stations.

The mapping architecture is:

BTS Airport Code
→ FAA Airport Record
→ Airport Latitude / Longitude
→ Candidate NOAA Stations
→ Nearest Suitable Weather Station

In [8]:
FAA_AIRPORT_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "airports"
    / "APT_BASE.csv"
)

faa_airports = pd.read_csv(
    FAA_AIRPORT_PATH,
    low_memory=False
)

print("FAA airport records:", len(faa_airports))
print("Columns:", len(faa_airports.columns))

faa_airports.head()

FAA airport records: 19426
Columns: 90


,EFF_DATE,SITE_NO,SITE_TYPE_CODE,STATE_CODE,ARPT_ID,CITY,COUNTRY_CODE,REGION_CODE,ADO_CODE,STATE_NAME,...,CONTR_FUEL_AVBL,TRNS_STRG_BUOY_FLAG,TRNS_STRG_HGR_FLAG,TRNS_STRG_TIE_FLAG,OTHER_SERVICES,WIND_INDCR_FLAG,ICAO_ID,MIN_OP_NETWORK,USER_FEE_FLAG,CTA
0,2026/08/06,103.00,A,AL,0J0,ABBEVILLE,US,ASO,JAN,ALABAMA,...,NaN,NaN,NaN,Y,NaN,Y-L,NaN,N,NaN,NaN
1,2026/08/06,106.00,A,AL,2A8,ADDISON,US,ASO,JAN,ALABAMA,...,NaN,NaN,NaN,N,NaN,Y,NaN,N,NaN,NaN
2,2026/08/06,108.00,A,AL,AL03,AKRON,US,ASO,JAN,ALABAMA,...,NaN,NaN,NaN,NaN,NaN,Y,NaN,N,NaN,NaN
3,2026/08/06,110.00,A,AL,EET,ALABASTER,US,ASO,JAN,ALABAMA,...,NaN,NaN,NaN,Y,"INSTR,RNTL,SALES",Y-L,KEET,N,NaN,NaN
4,2026/08/06,110.01,H,AL,AL39,ALABASTER,US,ASO,JAN,ALABAMA,...,NaN,NaN,NaN,NaN,NaN,Y,NaN,N,NaN,NaN


In [9]:
faa_airports.columns.tolist()

['EFF_DATE',
 'SITE_NO',
 'SITE_TYPE_CODE',
 'STATE_CODE',
 'ARPT_ID',
 'CITY',
 'COUNTRY_CODE',
 'REGION_CODE',
 'ADO_CODE',
 'STATE_NAME',
 'COUNTY_NAME',
 'COUNTY_ASSOC_STATE',
 'ARPT_NAME',
 'OWNERSHIP_TYPE_CODE',
 'FACILITY_USE_CODE',
 'LAT_DEG',
 'LAT_MIN',
 'LAT_SEC',
 'LAT_HEMIS',
 'LAT_DECIMAL',
 'LONG_DEG',
 'LONG_MIN',
 'LONG_SEC',
 'LONG_HEMIS',
 'LONG_DECIMAL',
 'SURVEY_METHOD_CODE',
 'ELEV',
 'ELEV_METHOD_CODE',
 'MAG_VARN',
 'MAG_HEMIS',
 'MAG_VARN_YEAR',
 'TPA',
 'CHART_NAME',
 'DIST_CITY_TO_AIRPORT',
 'DIRECTION_CODE',
 'ACREAGE',
 'RESP_ARTCC_ID',
 'COMPUTER_ID',
 'ARTCC_NAME',
 'FSS_ON_ARPT_FLAG',
 'FSS_ID',
 'FSS_NAME',
 'PHONE_NO',
 'TOLL_FREE_NO',
 'ALT_FSS_ID',
 'ALT_FSS_NAME',
 'ALT_TOLL_FREE_NO',
 'NOTAM_ID',
 'NOTAM_FLAG',
 'ACTIVATION_DATE',
 'ARPT_STATUS',
 'FAR_139_TYPE_CODE',
 'FAR_139_CARRIER_SER_CODE',
 'ARFF_CERT_TYPE_DATE',
 'NASP_CODE',
 'ASP_ANLYS_DTRM_CODE',
 'CUST_FLAG',
 'LNDG_RIGHTS_FLAG',
 'JOINT_USE_FLAG',
 'MIL_LNDG_FLAG',
 'INSPECT_METHOD_COD

## FAA Airport Coordinate Reference

The FAA airport base table contains the geographic reference fields required for weather-station matching.

The integration uses:

- `ARPT_ID` as the FAA airport identifier
- `ICAO_ID` as the ICAO identifier where available
- `LAT_DECIMAL` and `LONG_DECIMAL` as airport reference-point coordinates
- Airport name, city, and state fields for validation

The FAA table is reduced to these fields before matching against the BTS airport master list.

In [10]:
faa_airport_ref = faa_airports[
    [
        "ARPT_ID",
        "ICAO_ID",
        "ARPT_NAME",
        "CITY",
        "STATE_CODE",
        "LAT_DECIMAL",
        "LONG_DECIMAL"
    ]
].copy()

faa_airport_ref.head()

,ARPT_ID,ICAO_ID,ARPT_NAME,CITY,STATE_CODE,LAT_DECIMAL,LONG_DECIMAL
0,0J0,NaN,ABBEVILLE MUNI,ABBEVILLE,AL,31.601720,-85.238545
1,2A8,NaN,ADDISON,ADDISON,AL,34.217142,-87.158158
2,AL03,NaN,STRICKLAND/SMALLEY FLD,AKRON,AL,32.847500,-87.713889
3,EET,KEET,SHELBY COUNTY,ALABASTER,AL,33.177778,-86.783222
4,AL39,NaN,SHELBY MEDICAL CENTER,ALABASTER,AL,33.252222,-86.812222


In [11]:
faa_airport_ref[
    ["LAT_DECIMAL", "LONG_DECIMAL"]
].isna().sum()

LAT_DECIMAL     0
LONG_DECIMAL    0
dtype: int64

In [12]:
faa_airport_ref[
    faa_airport_ref["ARPT_ID"] == "MEM"
]

,ARPT_ID,ICAO_ID,ARPT_NAME,CITY,STATE_CODE,LAT_DECIMAL,LONG_DECIMAL
14323,MEM,KMEM,FREDERICK W SMITH INTL/MEMPHIS,MEMPHIS,TN,35.042411,-89.976679


In [13]:
airport_match = airport_master.merge(
    faa_airport_ref,
    left_on="airport",
    right_on="ARPT_ID",
    how="left"
)

airport_match.head()

,airport,bts_airport_id,city,state,ARPT_ID,ICAO_ID,ARPT_NAME,CITY,STATE_CODE,LAT_DECIMAL,LONG_DECIMAL
0,ABE,10135,"Allentown/Bethlehem/Easton, PA",PA,ABE,KABE,LEHIGH VALLEY INTL,ALLENTOWN,PA,40.652363,-75.440406
1,ABI,10136,"Abilene, TX",TX,ABI,KABI,ABILENE RGNL,ABILENE,TX,32.411333,-99.681889
2,ABQ,10140,"Albuquerque, NM",NM,ABQ,KABQ,ALBUQUERQUE INTL SUNPORT,ALBUQUERQUE,NM,35.038932,-106.608262
3,ABR,10141,"Aberdeen, SD",SD,ABR,KABR,ABERDEEN RGNL,ABERDEEN,SD,45.446798,-98.422441
4,ABY,10146,"Albany, GA",GA,ABY,KABY,SOUTHWEST GEORGIA RGNL,ALBANY,GA,31.535530,-84.194483


In [14]:
matched = airport_match["LAT_DECIMAL"].notna().sum()
total = len(airport_match)

print(f"Matched airports: {matched}/{total}")
print(f"Match rate: {matched / total * 100:.2f}%")

Matched airports: 341/352
Match rate: 96.88%


In [15]:
unmatched_airports = airport_match[
    airport_match["LAT_DECIMAL"].isna()
][
    ["airport", "city", "state"]
]

unmatched_airports

,airport,city,state
26,AZA,"Phoenix, AZ",AZ
67,CLD,"Carlsbad, CA",CA
117,FCA,"Kalispell, MT",MT
144,GUF,"Gulf Shores, AL",AL
148,HHH,"Hilton Head, SC",SC
226,MQT,"Marquette, MI",MI
249,PBI,"West Palm Beach/Palm Beach, FL",FL
297,SCE,"State College, PA",PA
317,SPN,"Saipan, TT",TT
342,USA,"Concord, NC",NC


### Airport Identifier Reconciliation

Direct matching between BTS airport codes and FAA location identifiers successfully matched 341 of 352 airports (96.88%).

The remaining airports require identifier reconciliation because BTS airline-service codes do not always correspond directly to FAA location identifiers. These cases will be handled through an explicit crosswalk rather than fuzzy city-name matching.

This preserves a transparent and auditable airport mapping process and prevents incorrect geographic assignments in the subsequent NOAA weather-station matching stage.

In [17]:
for code in unmatched_airports["airport"]:
    matches = faa_airport_ref[
        faa_airport_ref["ICAO_ID"]
        .fillna("")
        .str.endswith(code)
    ]

    if len(matches) > 0:
        print(f"\nBTS: {code}")
        display(
            matches[
                [
                    "ARPT_ID",
                    "ICAO_ID",
                    "ARPT_NAME",
                    "CITY",
                    "STATE_CODE",
                    "LAT_DECIMAL",
                    "LONG_DECIMAL"
                ]
            ]
        )

matches



,ARPT_ID,ICAO_ID,ARPT_NAME,CITY,STATE_CODE,LAT_DECIMAL,LONG_DECIMAL


In [18]:
for _, row in unmatched_airports.iterrows():

    bts_code = row["airport"]
    bts_city = row["city"].split(",")[0].strip()
    bts_state = row["state"]

    candidates = faa_airport_ref[
        (faa_airport_ref["STATE_CODE"] == bts_state)
        &
        (
            faa_airport_ref["CITY"]
            .fillna("")
            .str.contains(
                bts_city,
                case=False,
                regex=False
            )
        )
    ]

    print(
        f"\n--- BTS {bts_code}: "
        f"{row['city']} ---"
    )

    display(
        candidates[
            [
                "ARPT_ID",
                "ICAO_ID",
                "ARPT_NAME",
                "CITY",
                "STATE_CODE",
                "LAT_DECIMAL",
                "LONG_DECIMAL"
            ]
        ]
    )


--- BTS AZA: Phoenix, AZ ---


,ARPT_ID,ICAO_ID,ARPT_NAME,CITY,STATE_CODE,LAT_DECIMAL,LONG_DECIMAL
503,4AZ5,NaN,NEW WADDELL DAM,PHOENIX,AZ,33.845153,-112.269836
504,AZ07,NaN,PHOENIX AREA,PHOENIX,AZ,33.442733,-112.149150
505,AZ20,NaN,WESTCOR HOME OFFICE,PHOENIX,AZ,33.593189,-111.978981
506,AZ24,NaN,PHOENIX BAPTIST HOSPITAL,PHOENIX,AZ,33.525078,-112.101714
507,67AZ,NaN,BANNER UNI MED CENTER PHX,PHOENIX,AZ,33.465043,-112.060148
508,5AZ2,NaN,SOUTHERN COMMAND POLICE STATION,PHOENIX,AZ,33.415033,-112.072356
509,AZ29,NaN,WESTRIDGE MALL,PHOENIX,AZ,33.475598,-112.223765
510,AZ33,NaN,KNOELL-MAIN OFFICE,PHOENIX,AZ,33.425878,-112.029869
511,20E,NaN,MARICOPA MEDICAL CENTER,PHOENIX,AZ,33.456800,-112.026997
512,AZ48,NaN,BANNER UNIV MED CENTER,PHOENIX,AZ,33.465043,-112.057926



--- BTS CLD: Carlsbad, CA ---


,ARPT_ID,ICAO_ID,ARPT_NAME,CITY,STATE_CODE,LAT_DECIMAL,LONG_DECIMAL
1091,CRQ,KCRQ,MC CLELLAN-PALOMAR,CARLSBAD,CA,33.12825,-117.280083



--- BTS FCA: Kalispell, MT ---


...

,ARPT_ID,ICAO_ID,ARPT_NAME,CITY,STATE_CODE,LAT_DECIMAL,LONG_DECIMAL



--- BTS USA: Concord, NC ---


,ARPT_ID,ICAO_ID,ARPT_NAME,CITY,STATE_CODE,LAT_DECIMAL,LONG_DECIMAL
10904,4NC8,NaN,BUFFALO CREEK,CONCORD,NC,35.422362,-80.620623
10905,NC41,NaN,HENDRICK MOTORSPORTS,CONCORD,NC,35.357533,-80.705836
10906,NC60,NaN,CHS NORTHEAST MEDICAL CENTER,CONCORD,NC,35.435111,-80.601389
10907,NC19,NaN,PROPST,CONCORD,NC,35.391807,-80.575622
10908,NC77,NaN,CHALFANT,CONCORD,NC,35.455695,-80.575622
10909,JQF,KJQF,CONCORD-PADGETT RGNL,CONCORD,NC,35.387770,-80.709132
10910,8NC5,NaN,WB,CONCORD,NC,35.406667,-80.515000



--- BTS YUM: Yuma, AZ ---


,ARPT_ID,ICAO_ID,ARPT_NAME,CITY,STATE_CODE,LAT_DECIMAL,LONG_DECIMAL
654,NYL,KNYL,YUMA MCAS/YUMA INTL,YUMA,AZ,32.656574,-114.605987
655,05AZ,NaN,YUMA RGNL MEDICAL CENTER,YUMA,AZ,32.683622,-114.635058
656,34AZ,NaN,ONVIDA HEALTH FOOTHILLS MEDICAL PLAZA,YUMA,AZ,32.668611,-114.439306
657,LGF,KLGF,LAGUNA AAF (YUMA PROVING GROUND),YUMA PROVING GROUND (YUMA),AZ,32.864581,-114.392975


In [19]:
for code in ["PBI", "SPN"]:
    print(f"\n=== Searching for BTS {code} ===")

    candidates = faa_airport_ref[
        faa_airport_ref["ARPT_NAME"]
        .fillna("")
        .str.contains(
            "PALM BEACH|SAIPAN",
            case=False,
            regex=True
        )
    ]

    display(
        candidates[
            [
                "ARPT_ID",
                "ICAO_ID",
                "ARPT_NAME",
                "CITY",
                "STATE_CODE",
                "LAT_DECIMAL",
                "LONG_DECIMAL"
            ]
        ]
    )


=== Searching for BTS PBI ===


,ARPT_ID,ICAO_ID,ARPT_NAME,CITY,STATE_CODE,LAT_DECIMAL,LONG_DECIMAL
2893,42FD,NaN,PALM BEACH STATE COLLEGE,LAKE WORTH,FL,26.610217,-80.088103
3128,PHK,KPHK,PALM BEACH COUNTY GLADES,PAHOKEE,FL,26.785028,-80.693361
3134,87FD,NaN,PALM BEACH GARDENS MEDICAL CENTER,PALM BEACH GARDENS,FL,26.828692,-80.085369
3343,F45,NaN,NORTH PALM BEACH COUNTY GENERAL AVIATION,WEST PALM BEACH,FL,26.845917,-80.222333
3345,LNA,KLNA,PALM BEACH COUNTY PARK,WEST PALM BEACH,FL,26.593046,-80.085064
3349,39FL,NaN,PALM BEACH SHERIFF'S RANGE,WEST PALM BEACH,FL,26.717009,-80.199767
3350,5FL5,NaN,PALM BEACH COUNTY JUDICIAL CENTER,WEST PALM BEACH,FL,26.715408,-80.054400
3351,7FL5,NaN,WEST PALM BEACH POLICE STATION,WEST PALM BEACH,FL,26.714283,-80.058306
19200,GSN,PGSN,FRANCISCO C ADA/SAIPAN INTL,SAIPAN ISLAND,MP,15.120249,145.729986



=== Searching for BTS SPN ===


,ARPT_ID,ICAO_ID,ARPT_NAME,CITY,STATE_CODE,LAT_DECIMAL,LONG_DECIMAL
2893,42FD,NaN,PALM BEACH STATE COLLEGE,LAKE WORTH,FL,26.610217,-80.088103
3128,PHK,KPHK,PALM BEACH COUNTY GLADES,PAHOKEE,FL,26.785028,-80.693361
3134,87FD,NaN,PALM BEACH GARDENS MEDICAL CENTER,PALM BEACH GARDENS,FL,26.828692,-80.085369
3343,F45,NaN,NORTH PALM BEACH COUNTY GENERAL AVIATION,WEST PALM BEACH,FL,26.845917,-80.222333
3345,LNA,KLNA,PALM BEACH COUNTY PARK,WEST PALM BEACH,FL,26.593046,-80.085064
3349,39FL,NaN,PALM BEACH SHERIFF'S RANGE,WEST PALM BEACH,FL,26.717009,-80.199767
3350,5FL5,NaN,PALM BEACH COUNTY JUDICIAL CENTER,WEST PALM BEACH,FL,26.715408,-80.054400
3351,7FL5,NaN,WEST PALM BEACH POLICE STATION,WEST PALM BEACH,FL,26.714283,-80.058306
19200,GSN,PGSN,FRANCISCO C ADA/SAIPAN INTL,SAIPAN ISLAND,MP,15.120249,145.729986


### BTS–FAA Identifier Crosswalk

Direct BTS-to-FAA identifier matching covered 341 of 352 airports.

Eleven airports use BTS/IATA identifiers that differ from the corresponding FAA location identifiers or, in the case of Palm Beach, changed after the 2025 BTS observation period.

These exceptions are handled through an explicit crosswalk rather than fuzzy geographic matching. This preserves the historical BTS identifiers while allowing the current FAA airport reference table to supply authoritative airport coordinates.

In [20]:
airport_identifier_crosswalk = pd.DataFrame({
    "airport": [
        "AZA", "CLD", "FCA", "GUF", "HHH",
        "MQT", "PBI", "SCE", "SPN", "USA", "YUM"
    ],
    "faa_arpt_id": [
        "IWA", "CRQ", "GPI", "JKA", "HXD",
        "SAW", "DJT", "UNV", "GSN", "JQF", "NYL"
    ]
})

airport_identifier_crosswalk

,airport,faa_arpt_id
0,AZA,IWA
1,CLD,CRQ
2,FCA,GPI
3,GUF,JKA
4,HHH,HXD
5,MQT,SAW
6,PBI,DJT
7,SCE,UNV
8,SPN,GSN
9,USA,JQF


In [21]:
airport_master_mapped = airport_master.merge(
    airport_identifier_crosswalk,
    on="airport",
    how="left"
)

# Normal case: BTS code = FAA identifier
# Exception: use explicit crosswalk
airport_master_mapped["faa_match_id"] = (
    airport_master_mapped["faa_arpt_id"]
    .fillna(airport_master_mapped["airport"])
)

airport_geo = airport_master_mapped.merge(
    faa_airport_ref,
    left_on="faa_match_id",
    right_on="ARPT_ID",
    how="left"
)

In [22]:
matched = airport_geo["LAT_DECIMAL"].notna().sum()
total = len(airport_geo)

print(f"Matched airports: {matched}/{total}")
print(f"Match rate: {matched / total * 100:.2f}%")

Matched airports: 352/352
Match rate: 100.00%


In [23]:
airport_geo[
    [
        "airport",
        "faa_match_id",
        "ARPT_NAME",
        "CITY",
        "STATE_CODE",
        "LAT_DECIMAL",
        "LONG_DECIMAL"
    ]
].tail(15)

,airport,faa_match_id,ARPT_NAME,CITY,STATE_CODE,LAT_DECIMAL,LONG_DECIMAL
337,TVC,TVC,CHERRY CAPITAL,TRAVERSE CITY,MI,44.741579,-85.581870
338,TWF,TWF,JOSLIN FLD/MAGIC VALLEY RGNL,TWIN FALLS,ID,42.481809,-114.487736
339,TXK,TXK,TEXARKANA RGNL-WEBB FLD,TEXARKANA,AR,33.453716,-93.991031
340,TYR,TYR,TYLER POUNDS RGNL,TYLER,TX,32.353547,-95.402978
341,TYS,TYS,MC GHEE TYSON,KNOXVILLE,TN,35.811091,-83.994067
342,USA,JQF,CONCORD-PADGETT RGNL,CONCORD,NC,35.387770,-80.709132
343,VCT,VCT,VICTORIA RGNL,VICTORIA,TX,28.854206,-96.918732
344,VPS,VPS,EGLIN AFB/DESTIN-FT WALTON BEACH,VALPARAISO/DESTIN-FT WALTON BEACH,FL,30.483219,-86.526044
345,VRB,VRB,VERO BEACH RGNL,VERO BEACH,FL,27.655555,-80.417954
346,WRG,WRG,WRANGELL,WRANGELL,AK,56.484333,-132.369833


## NOAA Station Mapping Strategy

The flight dataset now contains validated geographic coordinates for all BTS airports.

The next stage maps each airport to an appropriate NOAA GHCNh weather station.

Rather than manually assigning stations, the mapping process will be automated using geographic distance between airport reference-point coordinates and NOAA station coordinates.

The station-selection workflow is:

1. Load NOAA GHCNh station metadata.
2. Restrict candidate stations to those with appropriate geographic and temporal coverage.
3. Calculate airport-to-station distance.
4. Select the nearest suitable station for each airport.
5. Audit unusually large airport-to-station distances.
6. Preserve the resulting airport–station crosswalk as a reusable project artifact.

This design provides a reproducible mapping process that can be rerun if NOAA station metadata changes.

## NOAA GHCNh Station Metadata

Airport coordinates are now available for all BTS airports.

The next step is to identify an appropriate NOAA GHCNh station for each airport. Station selection will be automated using geographic proximity and station coverage rather than manual assignment.

The mapping process will:

1. Load NOAA GHCNh station metadata.
2. Retain stations with valid coordinates and 2025 coverage.
3. Calculate geographic distance from each airport to candidate stations.
4. Select the nearest eligible station.
5. Flag unusually distant mappings for manual review.
6. Save the validated airport–station crosswalk as a reusable project artifact.

Hourly weather observations will not be downloaded until the station mapping has been validated.

In [24]:
STATION_LIST_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "weather"
    / "stations"
    / "ghcnh-station-list.txt"
)

with open(STATION_LIST_PATH, "r") as f:
    for _ in range(10):
        print(repr(f.readline()))

'ACL000BARA9  17.5910  -61.8210    5.0 TX BARBUDA                                      AG\n'
'ACM00078861  17.1167  -61.7833   10.0    COOLIDGE FIELD   ANTIGUA (AUX.         78861 AG\n'
'ACU55-00189  18.6000  -63.4700   10.0    SOMBRERO                                     AG\n'
'ACW00011647  17.1333  -61.7833   19.2    ST JOHNS                                     AG\n'
'AEI0000OMAA  24.4330   54.6511   26.8    ABU DHABI INTL                               AE\n'
'AEI0000OMAB  23.6167   53.3833   94.0    BUHASA                                       AE\n'
'AEI0000OMAD  24.4283   54.4581    4.9    BATEEN                                       AE\n'
'AEI0000OMAH  24.0740   52.4636   15.2    AL HAMRA AUX                                 AE\n'
'AEI0000OMAJ  24.1874   52.6140   13.1    JEBEL DHANA                                  AE\n'
'AEI0000OMAL  24.2617   55.6092  264.9    AL AIN INTL                                  AE\n'


### Parse GHCNh Station Metadata

The NOAA GHCNh station list is distributed as a fixed-width text file rather than a standard CSV.

The raw metadata is parsed into structured station attributes so that station coordinates can be used for automated geographic matching with BTS airports. The original NOAA file is retained unchanged in the raw data layer.

In [25]:
import pandas as pd

ghcnh_stations = pd.read_fwf(
    STATION_LIST_PATH,
    colspecs=[
        (0, 11),    # station_id
        (12, 20),   # latitude
        (21, 30),   # longitude
        (31, 37),   # elevation
        (38, 68),   # station_name
        (69, 72),   # wmo_id / auxiliary identifier
        (73, 75),   # country_code
    ],
    names=[
        "station_id",
        "latitude",
        "longitude",
        "elevation_m",
        "station_name",
        "wmo_id",
        "country_code",
    ]
)

ghcnh_stations.head()

,station_id,latitude,longitude,elevation_m,station_name,wmo_id,country_code
0,ACL000BARA9,17.5910,-61.8210,5.0,TX BARBUDA,NaN,NaN
1,ACM00078861,17.1167,-61.7833,10.0,COOLIDGE FIELD ANTIGUA (A,X.,NaN
2,ACU55-00189,18.6000,-63.4700,10.0,SOMBRERO,NaN,NaN
3,ACW00011647,17.1333,-61.7833,19.2,ST JOHNS,NaN,NaN
4,AEI0000OMAA,24.4330,54.6511,26.8,ABU DHABI INTL,NaN,NaN


In [26]:
print("Stations:", len(ghcnh_stations))

ghcnh_stations[
    [
        "station_id",
        "latitude",
        "longitude",
        "elevation_m",
        "station_name",
        "country_code"
    ]
].head(10)

Stations: 38870


,station_id,latitude,longitude,elevation_m,station_name,country_code
0,ACL000BARA9,17.5910,-61.8210,5.0,TX BARBUDA,NaN
1,ACM00078861,17.1167,-61.7833,10.0,COOLIDGE FIELD ANTIGUA (A,NaN
2,ACU55-00189,18.6000,-63.4700,10.0,SOMBRERO,NaN
3,ACW00011647,17.1333,-61.7833,19.2,ST JOHNS,NaN
4,AEI0000OMAA,24.4330,54.6511,26.8,ABU DHABI INTL,NaN
5,AEI0000OMAB,23.6167,53.3833,94.0,BUHASA,NaN
6,AEI0000OMAD,24.4283,54.4581,4.9,BATEEN,NaN
7,AEI0000OMAH,24.0740,52.4636,15.2,AL HAMRA AUX,NaN
8,AEI0000OMAJ,24.1874,52.6140,13.1,JEBEL DHANA,NaN
9,AEI0000OMAL,24.2617,55.6092,264.9,AL AIN INTL,NaN


In [27]:
ghcnh_stations = pd.read_fwf(
    STATION_LIST_PATH,
    colspecs=[
        (0, 11),
        (12, 20),
        (21, 30),
        (31, 37),
        (38, 68),
    ],
    names=[
        "station_id",
        "latitude",
        "longitude",
        "elevation_m",
        "station_name",
    ]
)

ghcnh_stations.head(10)

,station_id,latitude,longitude,elevation_m,station_name
0,ACL000BARA9,17.5910,-61.8210,5.0,TX BARBUDA
1,ACM00078861,17.1167,-61.7833,10.0,COOLIDGE FIELD ANTIGUA (A
2,ACU55-00189,18.6000,-63.4700,10.0,SOMBRERO
3,ACW00011647,17.1333,-61.7833,19.2,ST JOHNS
4,AEI0000OMAA,24.4330,54.6511,26.8,ABU DHABI INTL
5,AEI0000OMAB,23.6167,53.3833,94.0,BUHASA
6,AEI0000OMAD,24.4283,54.4581,4.9,BATEEN
7,AEI0000OMAH,24.0740,52.4636,15.2,AL HAMRA AUX
8,AEI0000OMAJ,24.1874,52.6140,13.1,JEBEL DHANA
9,AEI0000OMAL,24.2617,55.6092,264.9,AL AIN INTL


In [28]:
ghcnh_stations = ghcnh_stations.dropna(
    subset=["latitude", "longitude"]
).copy()

print("Stations with valid coordinates:", len(ghcnh_stations))

Stations with valid coordinates: 38870


## Automated Airport-to-NOAA Station Mapping

Each BTS airport is matched to the geographically nearest GHCNh weather station using airport and station latitude/longitude coordinates.

The initial match is based on geographic proximity only. The resulting distances are then audited so that unusually distant station assignments can be reviewed before hourly weather data are downloaded.

This separates automated matching from quality assurance: normal mappings are accepted programmatically, while only questionable cases require manual inspection.

In [29]:
from sklearn.neighbors import BallTree
import numpy as np

In [30]:
station_coords_rad = np.radians(
    ghcnh_stations[
        ["latitude", "longitude"]
    ].to_numpy()
)

airport_coords_rad = np.radians(
    airport_geo[
        ["LAT_DECIMAL", "LONG_DECIMAL"]
    ].to_numpy()
)

In [31]:
station_tree = BallTree(
    station_coords_rad,
    metric="haversine"
)

In [32]:
distances, indices = station_tree.query(
    airport_coords_rad,
    k=1
)

In [33]:
EARTH_RADIUS_MILES = 3958.7613

airport_station_map = airport_geo[
    [
        "airport",
        "ARPT_NAME",
        "LAT_DECIMAL",
        "LONG_DECIMAL"
    ]
].copy()

airport_station_map["station_index"] = indices[:, 0]

airport_station_map["station_distance_miles"] = (
    distances[:, 0] * EARTH_RADIUS_MILES
)

In [34]:
nearest_stations = (
    ghcnh_stations
    .iloc[indices[:, 0]]
    .reset_index(drop=True)
)

airport_station_map[
    "ghcnh_station_id"
] = nearest_stations["station_id"]

airport_station_map[
    "ghcnh_station_name"
] = nearest_stations["station_name"]

airport_station_map[
    "station_latitude"
] = nearest_stations["latitude"]

airport_station_map[
    "station_longitude"
] = nearest_stations["longitude"]

In [35]:
airport_station_map.sort_values(
    "station_distance_miles",
    ascending=False
).head(20)

,airport,ARPT_NAME,LAT_DECIMAL,LONG_DECIMAL,station_index,station_distance_miles,ghcnh_station_id,ghcnh_station_name,station_latitude,station_longitude
267,PSE,MERCEDITA,18.008778,-66.564528,20742,2.839110,RQC00667292,PR PONCE 4 E,18.0258,-66.5252
41,BMI,CENTRAL IL RGNL/BLOOMINGTON-NORMAL,40.477111,-88.915917,33293,1.841568,USW00054831,IL BLOOMINGTON NORMAL AP,40.4833,-88.9500
219,MIA,MIAMI INTL,25.795361,-80.290116,31919,1.740145,USW00012839,FL MIAMI INTL AP,25.7881,-80.3169
97,DTW,DETROIT METRO WAYNE COUNTY,42.212431,-83.353393,33919,1.721917,USW00094847,MI DETROIT METRO AP,42.2311,-83.3311
26,AZA,MESA GATEWAY,33.307824,-111.655459,32524,1.536545,USW00023104,AZ PHOENIX,33.2892,-111.6700
137,GRR,GERALD R FORD INTL,42.880833,-85.522806,33926,1.441358,USW00094860,MI GRAND RAPIDS,42.8939,-85.5450
241,ONT,ONTARIO INTL,34.056014,-117.601187,31481,1.404823,USW00003102,CA ONTARIO INTL AP,34.0531,-117.5769
311,SLC,SALT LAKE CITY INTL,40.788393,-111.977773,32693,1.399302,USW00024127,UT SALT LAKE CITY INTL AP,40.7706,-111.9650
89,DEN,DENVER INTL,39.861667,-104.673167,31407,1.374343,USW00003017,CO DENVER INTL AP,39.8467,-104.6561
247,PAE,SEATTLE PAINE FLD INTL,47.904011,-122.280869,32750,1.336735,USW00024222,WA EVERETT SNOHOMISH CO AP,47.9233,-122.2831


### Airport–Weather Station Mapping Validation

Nearest-neighbor geographic matching produced strong spatial alignment between BTS airports and GHCNh weather stations.

The maximum airport-to-station distance is less than 3 miles, indicating that the selected weather stations provide highly localized weather observations for the airports in the modeling population.

Because all mappings fall within a narrow geographic radius, no distance-based manual overrides are required at this stage. Station temporal coverage will be validated separately before weather observations are integrated.

In [36]:
airport_station_map["station_distance_miles"].describe(
    percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
)

count    352.000000
mean       0.536352
std        0.347480
min        0.004977
50%        0.461964
75%        0.712095
90%        0.971084
95%        1.159473
99%        1.627377
max        2.839110
Name: station_distance_miles, dtype: float64

In [37]:
pd.DataFrame({
    "distance_band": [
        "<= 1 mile",
        "1-3 miles",
        "> 3 miles"
    ],
    "airports": [
        (airport_station_map["station_distance_miles"] <= 1).sum(),
        (
            (airport_station_map["station_distance_miles"] > 1) &
            (airport_station_map["station_distance_miles"] <= 3)
        ).sum(),
        (airport_station_map["station_distance_miles"] > 3).sum()
    ]
})

,distance_band,airports
0,<= 1 mile,320
1,1-3 miles,32
2,> 3 miles,0


### Station Mapping QA Result

All 352 BTS airports were successfully mapped to nearby GHCNh stations.

- 320 airports (90.9%) are within 1 mile of their selected station.
- 32 airports (9.1%) are between 1 and 3 miles.
- No airport is more than 3 miles from its selected station.
- Median station distance is approximately 0.46 miles.
- Maximum station distance is approximately 2.84 miles.

These results indicate strong spatial alignment between airport locations and the selected weather stations. No distance-based manual overrides are required.

The remaining validation step is to confirm that the selected stations contain observations during the 2025 flight-data period.

In [38]:
CHECKPOINT_PATH = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "weather"
    / "airport_station_candidates.parquet"
)

CHECKPOINT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

airport_station_map.to_parquet(
    CHECKPOINT_PATH,
    index=False
)

print(f"Saved: {CHECKPOINT_PATH}")

Saved: /Users/tseringgurung/Desktop/flight-operations-intelligence/data/interim/weather/airport_station_candidates.parquet


In [39]:
selected_station_ids = (
    airport_station_map["ghcnh_station_id"]
    .dropna()
    .unique()
)

print("Airports:", len(airport_station_map))
print("Unique GHCNh stations:", len(selected_station_ids))
print("Duplicate station assignments:", len(airport_station_map) - len(selected_station_ids))

Airports: 352
Unique GHCNh stations: 352
Duplicate station assignments: 0


In [40]:
selected_station_ids[:20]

array(['USW00014737', 'USW00013962', 'USW00023050', 'USW00014929',
       'USW00013869', 'USW00014756', 'USW00013959', 'USW00024283',
       'USW00013753', 'USW00025701', 'USW00025501', 'USW00013934',
       'USW00003820', 'USU070326-1', 'USW00014735', 'USW00094910',
       'USC00410213', 'USW00026451', 'USW00094849', 'USW00093073'],
      dtype=object)

In [41]:
station_files_2025 = pd.DataFrame({
    "station_id": selected_station_ids
})

station_files_2025["filename"] = (
    "GHCNh_"
    + station_files_2025["station_id"]
    + "_2025.parquet"
)

station_files_2025.head(10)

,station_id,filename
0,USW00014737,GHCNh_USW00014737_2025.parquet
1,USW00013962,GHCNh_USW00013962_2025.parquet
2,USW00023050,GHCNh_USW00023050_2025.parquet
3,USW00014929,GHCNh_USW00014929_2025.parquet
4,USW00013869,GHCNh_USW00013869_2025.parquet
5,USW00014756,GHCNh_USW00014756_2025.parquet
6,USW00013959,GHCNh_USW00013959_2025.parquet
7,USW00024283,GHCNh_USW00024283_2025.parquet
8,USW00013753,GHCNh_USW00013753_2025.parquet
9,USW00025701,GHCNh_USW00025701_2025.parquet


In [42]:
from pathlib import Path

WEATHER_RAW_DIR = Path("../data/raw/weather/ghcnh/2025")
WEATHER_RAW_DIR.mkdir(parents=True, exist_ok=True)

print(WEATHER_RAW_DIR)

../data/raw/weather/ghcnh/2025


In [49]:
BASE_URL = (
    "https://www.ncei.noaa.gov/oa/"
    "global-historical-climatology-network/"
    "hourly/access/by-year/2025/parquet/"
)

In [50]:
station_files_2025["url"] = (
    BASE_URL + station_files_2025["filename"]
)

In [51]:
KNOWN_FILE = "GHCNh_ACW00011647_2025.parquet"

known_url = BASE_URL + KNOWN_FILE
known_path = WEATHER_RAW_DIR / KNOWN_FILE

print(known_url)

urllib.request.urlretrieve(
    known_url,
    known_path
)

print(
    f"Downloaded successfully: "
    f"{known_path.stat().st_size / 1024:.1f} KB"
)

https://www.ncei.noaa.gov/oa/global-historical-climatology-network/hourly/access/by-year/2025/parquet/GHCNh_ACW00011647_2025.parquet
Downloaded successfully: 559.0 KB


In [52]:
test_row = station_files_2025.iloc[0]

test_url = test_row["url"]
test_path = WEATHER_RAW_DIR / test_row["filename"]

print(test_url)

urllib.request.urlretrieve(
    test_url,
    test_path
)

print(
    f"Downloaded successfully: "
    f"{test_path.stat().st_size / 1024:.1f} KB"
)

https://www.ncei.noaa.gov/oa/global-historical-climatology-network/hourly/access/by-year/2025/parquet/GHCNh_USW00014737_2025.parquet
Downloaded successfully: 1074.4 KB


### Validate 2025 Station Availability

Before bulk downloading hourly weather observations, each selected GHCNh station is checked for the presence of a 2025 Parquet file.

This separates temporal coverage validation from data acquisition and prevents unnecessary failed downloads.

Stations without 2025 coverage will be handled through a controlled fallback to the next-nearest eligible station.

In [53]:
from urllib.request import Request, urlopen
from urllib.error import HTTPError, URLError

def check_url_exists(url):
    request = Request(
        url,
        method="HEAD",
        headers={"User-Agent": "Mozilla/5.0"}
    )

    try:
        with urlopen(request, timeout=15) as response:
            return response.status == 200
    except (HTTPError, URLError):
        return False

In [54]:
station_files_2025["available_2025"] = (
    station_files_2025["url"]
    .apply(check_url_exists)
)

station_files_2025["available_2025"].value_counts()

available_2025
True     331
False     21
Name: count, dtype: int64

In [55]:
missing_station_ids = set(
    station_files_2025.loc[
        ~station_files_2025["available_2025"],
        "station_id"
    ]
)

affected_airports = airport_station_map[
    airport_station_map["ghcnh_station_id"].isin(missing_station_ids)
].copy()

print("Missing stations:", len(missing_station_ids))
print("Affected airports:", len(affected_airports))

affected_airports[
    [
        "airport",
        "ARPT_NAME",
        "ghcnh_station_id",
        "ghcnh_station_name",
        "station_distance_miles"
    ]
].sort_values("station_distance_miles")

Missing stations: 21
Affected airports: 21


,airport,ARPT_NAME,ghcnh_station_id,ghcnh_station_name,station_distance_miles
63,CHS,CHARLESTON AFB/INTL,USW00013758,SC CHARLESTON NAS,0.004977
293,SBA,SANTA BARBARA MUNI,USW00053153,CA SANTA BARBARA,0.068273
246,OTZ,RALPH WIEN MEML,USA00701334,AK KOTZEBUE,0.134579
226,MQT,MARQUETTE/SAWYER RGNL,USW00014851,MI MARQUETTE K I SAWYER AP,0.182554
145,GUM,GUAM INTL,GQW00041406,GU GUAM WFO,0.198060
86,DCA,RONALD REAGAN WASHINGTON NTL,USW00013751,MD ANACOSTIA NAS,0.257883
230,MSP,MINNEAPOLIS-ST PAUL INTL/WOLD-CHAMBERLAIN,USW00014947,MN MINNEAPOLIS NAS,0.264977
9,ADK,ADAK,USW00025701,AK ADAK DAVIS AFB,0.321219
129,GFK,GRAND FORKS INTL,USW00014917,ND GRAND FORKS INTL AP,0.381930
151,HNL,DANIEL K INOUYE INTL,USI0000PHIK,HI HICKAM AFB,0.427713


In [56]:
import numpy as np
from sklearn.neighbors import BallTree

# Coordinates of every GHCNh station
station_coords_rad = np.radians(
    ghcnh_stations[["latitude", "longitude"]].to_numpy()
)

station_tree = BallTree(
    station_coords_rad,
    metric="haversine"
)

EARTH_RADIUS_MILES = 3958.8
MAX_STATION_DISTANCE_MILES = 5
N_CANDIDATES = 15

In [57]:
def find_available_station_2025(airport_row):
    airport_coord = np.radians([[
        airport_row["LAT_DECIMAL"],
        airport_row["LONG_DECIMAL"]
    ]])

    distances, indices = station_tree.query(
        airport_coord,
        k=N_CANDIDATES
    )

    for distance_rad, station_idx in zip(
        distances[0],
        indices[0]
    ):
        station = ghcnh_stations.iloc[station_idx]
        distance_miles = distance_rad * EARTH_RADIUS_MILES

        # Don't accept increasingly distant weather proxies
        if distance_miles > MAX_STATION_DISTANCE_MILES:
            break

        station_id = station["station_id"]
        filename = f"GHCNh_{station_id}_2025.parquet"
        url = BASE_URL + filename

        if check_url_exists(url):
            return {
                "ghcnh_station_id": station_id,
                "ghcnh_station_name": station["station_name"],
                "station_latitude": station["latitude"],
                "station_longitude": station["longitude"],
                "station_distance_miles": distance_miles,
                "filename": filename,
                "url": url,
                "available_2025": True
            }

    return None

In [58]:
fallback_results = []

for _, airport_row in affected_airports.iterrows():

    result = find_available_station_2025(airport_row)

    if result is None:
        fallback_results.append({
            "airport": airport_row["airport"],
            "ARPT_NAME": airport_row["ARPT_NAME"],
            "replacement_found": False
        })

    else:
        fallback_results.append({
            "airport": airport_row["airport"],
            "ARPT_NAME": airport_row["ARPT_NAME"],
            "replacement_found": True,
            **result
        })

fallback_results = pd.DataFrame(fallback_results)

fallback_results[
    [
        "airport",
        "ARPT_NAME",
        "replacement_found",
        "ghcnh_station_id",
        "ghcnh_station_name",
        "station_distance_miles"
    ]
].sort_values(
    ["replacement_found", "station_distance_miles"],
    ascending=[True, True]
)

,airport,ARPT_NAME,replacement_found,ghcnh_station_id,ghcnh_station_name,station_distance_miles
19,SGU,ST GEORGE RGNL,False,NaN,NaN,NaN
6,CHS,CHARLESTON AFB/INTL,True,USW00013880,SC CHARLESTON INTL AP,0.054909
18,SBA,SANTA BARBARA MUNI,True,USW00023190,CA SANTA BARBARA MUNI AP,0.149105
12,MQT,MARQUETTE/SAWYER RGNL,True,USW00094836,MI GWINN K I SAWYER AFB,0.182556
10,GUM,GUAM INTL,True,GQW00041415,GU GUAM INTL AP,0.198062
15,OTZ,RALPH WIEN MEML,True,USW00026616,AK KOTZEBUE AP,0.238616
1,ADK,ADAK,True,USW00025704,AK ADAK AP,0.321222
8,DCA,RONALD REAGAN WASHINGTON NTL,True,USW00013743,VA WASHINGTON REAGAN NATL AP,0.343189
9,GFK,GRAND FORKS INTL,True,USW00014916,ND GRAND FORKS INTL AP,0.490307
13,MSP,MINNEAPOLIS-ST PAUL INTL/WOLD-CHAMBERLAIN,True,USW00014922,MN MINNEAPOLIS-ST PAUL INTL AP,0.524186


In [59]:
print(
    fallback_results["replacement_found"]
    .value_counts(dropna=False)
)

print("\nDistance summary for replacements:")

display(
    fallback_results.loc[
        fallback_results["replacement_found"],
        "station_distance_miles"
    ].describe(
        percentiles=[.50, .75, .90, .95, .99]
    )
)

replacement_found
True     20
False     1
Name: count, dtype: int64

Distance summary for replacements:


count    20.000000
mean      0.683887
std       0.462087
min       0.054909
50%       0.694532
75%       0.869898
90%       1.335320
95%       1.396088
99%       1.614427
max       1.669012
Name: station_distance_miles, dtype: float64

In [60]:
sgu = affected_airports.loc[
    affected_airports["airport"] == "SGU"
].iloc[0]

sgu_coord = np.radians([[
    sgu["LAT_DECIMAL"],
    sgu["LONG_DECIMAL"]
]])

distances, indices = station_tree.query(
    sgu_coord,
    k=30
)

sgu_candidates = ghcnh_stations.iloc[indices[0]].copy()

sgu_candidates["distance_miles"] = (
    distances[0] * EARTH_RADIUS_MILES
)

sgu_candidates[
    [
        "station_id",
        "station_name",
        "latitude",
        "longitude",
        "distance_miles"
    ]
].head(20)

,station_id,station_name,latitude,longitude,distance_miles
33553,USW00093198,UT ST GEORGE INTERM FLD,37.0500,-113.5167,1.005148
32570,USW00023186,UT ST GEORGE MUNI AP,37.1000,-113.6000,6.616521
29395,USC00427516,UT ST. GEORGE,37.1190,-113.6068,7.802870
33051,USW00053166,UT ST. GEORGE 15 NE,37.2158,-113.3794,14.341861
33022,USW00053129,AZ COLORADO CITY MUNI AP,36.9597,-113.0139,27.900308
29402,USC00429717,UT ZION NATIONAL PARK,37.2091,-112.9814,31.487344
33001,USW00053013,UT CEDAR CITY 18 SSE,37.4572,-113.2247,33.048254
29369,USC00421162,UT CANYON BREEZE RANCH,37.6661,-113.5423,43.545546
29370,USC00421260,UT CEDAR CITY 5E,37.6565,-112.9918,51.448681
33537,USW00093129,UT CEDAR CITY AP,37.7067,-113.0969,51.578874


In [61]:
sgu_station_id = "USW00023186"
sgu_filename = f"GHCNh_{sgu_station_id}_2025.parquet"

sgu_url = (
    "https://www.ncei.noaa.gov/oa/"
    "global-historical-climatology-network/hourly/"
    "access/by-year/2025/parquet/"
    + sgu_filename
)

print(sgu_url)

https://www.ncei.noaa.gov/oa/global-historical-climatology-network/hourly/access/by-year/2025/parquet/GHCNh_USW00023186_2025.parquet


In [62]:
import urllib.request
import urllib.error

try:
    req = urllib.request.Request(sgu_url, method="HEAD")
    
    with urllib.request.urlopen(req) as response:
        print("Available:", response.status == 200)
        print("HTTP status:", response.status)

except urllib.error.HTTPError as e:
    print("Available: False")
    print("HTTP status:", e.code)
    

Available: True
HTTP status: 200


### Final Airport-to-GHCNh Station Mapping

All 352 BTS airports now have a validated GHCNh station with 2025 hourly weather coverage.

The station-selection process used:

1. Nearest geographic GHCNh station.
2. Validation that a 2025 Parquet file exists.
3. Automatic fallback to the next-nearest station when the nearest station lacked 2025 coverage.
4. A targeted exception for SGU, where the nearest 2025-available station is approximately 6.62 miles away.

For 351 airports, the selected 2025 weather station is within approximately 1.7 miles. SGU is the only larger fallback and remains within a reasonable local weather radius.

This finalized crosswalk is persisted so station selection does not need to be recomputed in subsequent runs.

In [63]:
# Start from the original nearest-station mapping
final_airport_station_map = airport_station_map.copy()

# Apply the 20 automatic fallback replacements
for _, row in fallback_results[
    fallback_results["replacement_found"]
].iterrows():
    
    mask = final_airport_station_map["airport"] == row["airport"]
    
    final_airport_station_map.loc[mask, "ghcnh_station_id"] = row["ghcnh_station_id"]
    final_airport_station_map.loc[mask, "ghcnh_station_name"] = row["ghcnh_station_name"]
    final_airport_station_map.loc[mask, "station_latitude"] = row["station_latitude"]
    final_airport_station_map.loc[mask, "station_longitude"] = row["station_longitude"]
    final_airport_station_map.loc[mask, "station_distance_miles"] = row["station_distance_miles"]

# SGU manual fallback
sgu_mask = final_airport_station_map["airport"] == "SGU"

final_airport_station_map.loc[sgu_mask, "ghcnh_station_id"] = "USW00023186"
final_airport_station_map.loc[sgu_mask, "ghcnh_station_name"] = "UT ST GEORGE MUNI AP"
final_airport_station_map.loc[sgu_mask, "station_distance_miles"] = 6.616521

In [64]:
print("Airports:", len(final_airport_station_map))
print(
    "Unique selected stations:",
    final_airport_station_map["ghcnh_station_id"].nunique()
)

print(
    "Missing station IDs:",
    final_airport_station_map["ghcnh_station_id"].isna().sum()
)

print(
    "Maximum distance:",
    final_airport_station_map["station_distance_miles"].max()
)

Airports: 352
Unique selected stations: 352
Missing station IDs: 0
Maximum distance: 6.616521


In [65]:
FINAL_STATION_MAP_PATH = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "weather"
    / "airport_station_map_2025.parquet"
)

final_airport_station_map.to_parquet(
    FINAL_STATION_MAP_PATH,
    index=False
)

print(f"Saved: {FINAL_STATION_MAP_PATH}")

Saved: /Users/tseringgurung/Desktop/flight-operations-intelligence/data/interim/weather/airport_station_map_2025.parquet


## Download 2025 GHCNh Weather for Selected Stations

The airport-to-station mapping is now finalized and persisted.

The next stage downloads only the 2025 GHCNh Parquet files required by the selected airport stations. This avoids downloading the full NOAA archive and keeps the weather pipeline focused on the stations actually used by the flight network.

The download process will:

1. Build a manifest from the finalized unique station list.
2. Skip files already present locally.
3. Download only missing 2025 Parquet files.
4. Record download status for reproducibility.
5. Inspect the GHCNh schema before selecting operationally relevant weather variables.

In [66]:
final_station_ids = (
    final_airport_station_map["ghcnh_station_id"]
    .dropna()
    .unique()
)

weather_manifest = pd.DataFrame({
    "station_id": final_station_ids
})

weather_manifest["filename"] = (
    "GHCNh_"
    + weather_manifest["station_id"]
    + "_2025.parquet"
)

weather_manifest["url"] = (
    BASE_URL
    + weather_manifest["filename"]
)

weather_manifest["local_path"] = weather_manifest["filename"].apply(
    lambda x: str(WEATHER_RAW_DIR / x)
)

print("Airports:", len(final_airport_station_map))
print("Unique weather stations:", len(weather_manifest))

weather_manifest.head()

Airports: 352
Unique weather stations: 352


,station_id,filename,url,local_path
0,USW00014737,GHCNh_USW00014737_2025.parquet,https://www.ncei.noaa.gov/oa/global-historical...,../data/raw/weather/ghcnh/2025/GHCNh_USW000147...
1,USW00013962,GHCNh_USW00013962_2025.parquet,https://www.ncei.noaa.gov/oa/global-historical...,../data/raw/weather/ghcnh/2025/GHCNh_USW000139...
2,USW00023050,GHCNh_USW00023050_2025.parquet,https://www.ncei.noaa.gov/oa/global-historical...,../data/raw/weather/ghcnh/2025/GHCNh_USW000230...
3,USW00014929,GHCNh_USW00014929_2025.parquet,https://www.ncei.noaa.gov/oa/global-historical...,../data/raw/weather/ghcnh/2025/GHCNh_USW000149...
4,USW00013869,GHCNh_USW00013869_2025.parquet,https://www.ncei.noaa.gov/oa/global-historical...,../data/raw/weather/ghcnh/2025/GHCNh_USW000138...


In [67]:
import time
import urllib.request
import urllib.error

def download_station_file(row, retries=3):
    destination = Path(row["local_path"])

    # Resume-friendly: don't redownload existing files
    if destination.exists() and destination.stat().st_size > 0:
        return "already_exists"

    for attempt in range(1, retries + 1):
        try:
            urllib.request.urlretrieve(
                row["url"],
                destination
            )
            return "downloaded"

        except Exception as e:
            if attempt == retries:
                return f"failed: {type(e).__name__}"

            time.sleep(2 * attempt)

In [68]:
download_status = []

for i, row in weather_manifest.iterrows():

    status = download_station_file(row)

    download_status.append(status)

    if (i + 1) % 25 == 0 or i == len(weather_manifest) - 1:
        print(
            f"{i + 1}/{len(weather_manifest)} completed"
        )

weather_manifest["download_status"] = download_status

25/352 completed
50/352 completed
75/352 completed
100/352 completed
125/352 completed
150/352 completed
175/352 completed
200/352 completed
225/352 completed
250/352 completed
275/352 completed
300/352 completed
325/352 completed
350/352 completed
352/352 completed


In [69]:
weather_manifest["download_status"].value_counts()

download_status
downloaded        351
already_exists      1
Name: count, dtype: int64

In [70]:
weather_manifest[
    weather_manifest["download_status"].str.startswith("failed")
]

,station_id,filename,url,local_path,download_status


In [71]:
MANIFEST_PATH = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "weather"
    / "ghcnh_2025_download_manifest.parquet"
)

weather_manifest.to_parquet(
    MANIFEST_PATH,
    index=False
)

print(f"Saved: {MANIFEST_PATH}")

Saved: /Users/tseringgurung/Desktop/flight-operations-intelligence/data/interim/weather/ghcnh_2025_download_manifest.parquet


In [72]:
sample_weather_path = Path(
    weather_manifest.iloc[0]["local_path"]
)

sample_weather = pd.read_parquet(
    sample_weather_path
)

print("Shape:", sample_weather.shape)

print("\nColumns:")
print(sample_weather.columns.tolist())

Shape: (13565, 329)

Columns:
['STATION', 'Station_name', 'DATE', 'Year', 'Month', 'Day', 'Hour', 'Minute', 'LATITUDE', 'LONGITUDE', 'ELEVATION', 'temperature', 'temperature_Measurement_Code', 'temperature_Quality_Code', 'temperature_Report_Type', 'temperature_Source_Code', 'temperature_Source_Station_ID', 'dew_point_temperature', 'dew_point_temperature_Measurement_Code', 'dew_point_temperature_Quality_Code', 'dew_point_temperature_Report_Type', 'dew_point_temperature_Source_Code', 'dew_point_temperature_Source_Station_ID', 'station_level_pressure', 'station_level_pressure_Measurement_Code', 'station_level_pressure_Quality_Code', 'station_level_pressure_Report_Type', 'station_level_pressure_Source_Code', 'station_level_pressure_Source_Station_ID', 'sea_level_pressure', 'sea_level_pressure_Measurement_Code', 'sea_level_pressure_Quality_Code', 'sea_level_pressure_Report_Type', 'sea_level_pressure_Source_Code', 'sea_level_pressure_Source_Station_ID', 'wind_direction', 'wind_direction_Meas

In [73]:
sample_weather_path = Path(
    weather_manifest.iloc[0]["local_path"]
)

sample_weather = pd.read_parquet(sample_weather_path)

print("File:", sample_weather_path.name)
print("Rows:", len(sample_weather))
print("Columns:", len(sample_weather.columns))

print("\nColumn names:")
for col in sample_weather.columns:
    print(col)

File: GHCNh_USW00014737_2025.parquet
Rows: 13565
Columns: 329

Column names:
STATION
Station_name
DATE
Year
Month
Day
Hour
Minute
LATITUDE
LONGITUDE
ELEVATION
temperature
temperature_Measurement_Code
temperature_Quality_Code
temperature_Report_Type
temperature_Source_Code
temperature_Source_Station_ID
dew_point_temperature
dew_point_temperature_Measurement_Code
dew_point_temperature_Quality_Code
dew_point_temperature_Report_Type
dew_point_temperature_Source_Code
dew_point_temperature_Source_Station_ID
station_level_pressure
station_level_pressure_Measurement_Code
station_level_pressure_Quality_Code
station_level_pressure_Report_Type
station_level_pressure_Source_Code
station_level_pressure_Source_Station_ID
sea_level_pressure
sea_level_pressure_Measurement_Code
sea_level_pressure_Quality_Code
sea_level_pressure_Report_Type
sea_level_pressure_Source_Code
sea_level_pressure_Source_Station_ID
wind_direction
wind_direction_Measurement_Code
wind_direction_Quality_Code
wind_direction_Report_

In [74]:
sample_weather.head()

,STATION,Station_name,DATE,Year,Month,Day,Hour,Minute,LATITUDE,LONGITUDE,...,precipitation_24_hour_Quality_Code,precipitation_24_hour_Report_Type,precipitation_24_hour_Source_Code,precipitation_24_hour_Source_Station_ID,REM,REM_Measurement_Code,REM_Quality_Code,REM_Report_Type,REM_Source_Code,REM_Source_Station_ID
0,USW00014737,ALLENTOWN LEHIGH VLY INTL AP,2025-01-01T00:00:00,2025,01,01,00,00,40.6497,-75.4478,...,None,None,None,None,SYN08072517 32666 80813 10083 20011 39902 4004...,None,None,FM12,223,ICAO-KABE
1,USW00014737,ALLENTOWN LEHIGH VLY INTL AP,2025-01-01T00:51:00,2025,01,01,00,51,40.6497,-75.4478,...,None,None,None,None,MET14612/31/24 19:51:03 METAR KABE 010051Z 070...,None,None,FM15,343,725170-14737
2,USW00014737,ALLENTOWN LEHIGH VLY INTL AP,2025-01-01T01:51:00,2025,01,01,01,51,40.6497,-75.4478,...,None,None,None,None,MET12612/31/24 20:51:03 METAR KABE 010151Z 080...,None,None,FM15,343,725170-14737
3,USW00014737,ALLENTOWN LEHIGH VLY INTL AP,2025-01-01T02:12:00,2025,01,01,02,12,40.6497,-75.4478,...,None,None,None,None,MET13312/31/24 21:12:03 SPECI KABE 010212Z 050...,None,None,FM16,343,725170-14737
4,USW00014737,ALLENTOWN LEHIGH VLY INTL AP,2025-01-01T02:22:00,2025,01,01,02,22,40.6497,-75.4478,...,None,None,None,None,MET14712/31/24 21:22:03 SPECI KABE 010222Z VRB...,None,None,FM16,343,725170-14737


In [75]:
sample_weather.dtypes

STATION                  object
Station_name             object
DATE                     object
Year                     object
Month                    object
                          ...  
REM_Measurement_Code     object
REM_Quality_Code         object
REM_Report_Type          object
REM_Source_Code          object
REM_Source_Station_ID    object
Length: 329, dtype: object

In [76]:
# Show columns that are NOT metadata / quality / source fields

metadata_patterns = (
    "_Quality_Code",
    "_Report_Type",
    "_Source_Code",
    "_Source_Station_ID",
    "_Measurement_Code"
)

measurement_cols = [
    col for col in sample_weather.columns
    if not col.endswith(metadata_patterns)
]

print("Potential measurement columns:", len(measurement_cols))

for col in measurement_cols:
    print(col)

Potential measurement columns: 64
STATION
Station_name
DATE
Year
Month
Day
Hour
Minute
LATITUDE
LONGITUDE
ELEVATION
temperature
dew_point_temperature
station_level_pressure
sea_level_pressure
wind_direction
wind_speed
wind_gust
precipitation
relative_humidity
wet_bulb_temperature
pres_wx_MW1
pres_wx_MW2
pres_wx_MW3
pres_wx_AU1
pres_wx_AU2
pres_wx_AU3
pres_wx_AW1
pres_wx_AW2
pres_wx_AW3
snow_depth
visibility
altimeter
pressure_3hr_change
sky_condition
sky_condition_baseht
ceiling_height
sky_cover_layer_1
sky_cover_layer_2
sky_cover_layer_3
sky_cover_layer_4
sky_cover_layer_baseht_1
sky_cover_layer_baseht_2
sky_cover_layer_baseht_3
sky_cover_layer_baseht_4
sky_cover_summation_1
sky_cover_summation_2
sky_cover_summation_3
sky_cover_summation_4
sky_cover_summation_baseht_1
sky_cover_summation_baseht_2
sky_cover_summation_baseht_3
sky_cover_summation_baseht_4
precipitation_5_minute
precipitation_15_minute
precipitation_3_hour
precipitation_6_hour
precipitation_9_hour
precipitation_12_hour
p

In [77]:
keywords = [
    "temperature",
    "dew",
    "wind",
    "visibility",
    "precip",
    "rain",
    "snow",
    "pressure",
    "ceiling",
    "cloud"
]

for keyword in keywords:
    matches = [
        col for col in sample_weather.columns
        if keyword.lower() in col.lower()
    ]

    print(f"\n--- {keyword.upper()} ---")
    for col in matches:
        print(col)


--- TEMPERATURE ---
temperature
temperature_Measurement_Code
temperature_Quality_Code
temperature_Report_Type
temperature_Source_Code
temperature_Source_Station_ID
dew_point_temperature
dew_point_temperature_Measurement_Code
dew_point_temperature_Quality_Code
dew_point_temperature_Report_Type
dew_point_temperature_Source_Code
dew_point_temperature_Source_Station_ID
wet_bulb_temperature
wet_bulb_temperature_Measurement_Code
wet_bulb_temperature_Quality_Code
wet_bulb_temperature_Report_Type
wet_bulb_temperature_Source_Code
wet_bulb_temperature_Source_Station_ID

--- DEW ---
dew_point_temperature
dew_point_temperature_Measurement_Code
dew_point_temperature_Quality_Code
dew_point_temperature_Report_Type
dew_point_temperature_Source_Code
dew_point_temperature_Source_Station_ID

--- WIND ---
wind_direction
wind_direction_Measurement_Code
wind_direction_Quality_Code
wind_direction_Report_Type
wind_direction_Source_Code
wind_direction_Source_Station_ID
wind_speed
wind_speed_Measurement_Code
w

In [78]:
candidate_weather_cols = [
    "STATION",
    "DATE",
    "temperature",
    "dew_point_temperature",
    "wind_direction",
    "wind_speed",
    "wind_gust",
    "visibility",
    "sea_level_pressure",
    "station_level_pressure",
    "precipitation",
    "ceiling_height"
]

available_weather_cols = [
    col for col in candidate_weather_cols
    if col in sample_weather.columns
]

missing_weather_cols = [
    col for col in candidate_weather_cols
    if col not in sample_weather.columns
]

print("AVAILABLE:")
for col in available_weather_cols:
    print(col)

print("\nNOT FOUND:")
for col in missing_weather_cols:
    print(col)

AVAILABLE:
STATION
DATE
temperature
dew_point_temperature
wind_direction
wind_speed
wind_gust
visibility
sea_level_pressure
station_level_pressure
precipitation
ceiling_height

NOT FOUND:


In [79]:
sample_weather[
    available_weather_cols
].head(20)

,STATION,DATE,temperature,dew_point_temperature,wind_direction,wind_speed,wind_gust,visibility,sea_level_pressure,station_level_pressure,precipitation,ceiling_height
0,USW00014737,2025-01-01T00:00:00,8.3,1.1,080,6.7,NaN,16.0,1004.1,990.2,NaN,None
1,USW00014737,2025-01-01T00:51:00,8.3,3.9,070,6.7,NaN,16.093,1002.5,988.5,0.3,1219
2,USW00014737,2025-01-01T01:51:00,7.8,5.6,080,7.2,NaN,8.047,1000.5,986.5,4.3,1829
3,USW00014737,2025-01-01T02:12:00,7.8,6.1,050,7.7,NaN,8.047,NaN,985.5,2.2,1372
4,USW00014737,2025-01-01T02:22:00,7.8,6.1,999,1.5,NaN,4.023,NaN,986.8,3.5,1372
5,USW00014737,2025-01-01T02:29:00,7.8,6.1,030,2.1,NaN,3.219,NaN,985.8,6.8,488
6,USW00014737,2025-01-01T02:38:00,7.8,6.1,050,3.6,NaN,9.656,NaN,985.8,7.6,488
7,USW00014737,2025-01-01T02:41:00,7.8,6.1,040,4.1,NaN,9.656,NaN,985.8,7.8,274
8,USW00014737,2025-01-01T02:48:00,8.0,6.0,010,2.1,NaN,11.265,NaN,985.8,8.6,335
9,USW00014737,2025-01-01T02:51:00,7.8,6.1,010,2.1,NaN,14.484,999.9,985.8,7.9,335


In [80]:
coverage = pd.DataFrame({
    "column": available_weather_cols,
    "non_null": [
        sample_weather[col].notna().sum()
        for col in available_weather_cols
    ],
    "total_rows": len(sample_weather)
})

coverage["coverage_pct"] = (
    coverage["non_null"] /
    coverage["total_rows"] * 100
).round(2)

coverage

,column,non_null,total_rows,coverage_pct
0,STATION,13565,13565,100.00
1,DATE,13565,13565,100.00
2,temperature,13434,13565,99.03
3,dew_point_temperature,13442,13565,99.09
4,wind_direction,13421,13565,98.94
5,wind_speed,13417,13565,98.91
6,wind_gust,1613,13565,11.89
7,visibility,13448,13565,99.14
8,sea_level_pressure,11145,13565,82.16
9,station_level_pressure,10155,13565,74.86


In [81]:
inspect_cols = [
    "DATE",
    "temperature",
    "dew_point_temperature",
    "wind_direction",
    "wind_speed",
    "visibility",
    "sea_level_pressure",
    "precipitation",
    "ceiling_height"
]

for col in inspect_cols:
    print(f"\n--- {col} ---")
    print(sample_weather[col].dropna().head(10).tolist())


--- DATE ---
['2025-01-01T00:00:00', '2025-01-01T00:51:00', '2025-01-01T01:51:00', '2025-01-01T02:12:00', '2025-01-01T02:22:00', '2025-01-01T02:29:00', '2025-01-01T02:38:00', '2025-01-01T02:41:00', '2025-01-01T02:48:00', '2025-01-01T02:51:00']

--- temperature ---
[8.3, 8.3, 7.8, 7.8, 7.8, 7.8, 7.8, 7.8, 8.0, 7.8]

--- dew_point_temperature ---
[1.1, 3.9, 5.6, 6.1, 6.1, 6.1, 6.1, 6.1, 6.0, 6.1]

--- wind_direction ---
['080', '070', '080', '050', '999', '030', '050', '040', '010', '010']

--- wind_speed ---
[6.7, 6.7, 7.2, 7.7, 1.5, 2.1, 3.6, 4.1, 2.1, 2.1]

--- visibility ---
['16.0', '16.093', '8.047', '8.047', '4.023', '3.219', '9.656', '9.656', '11.265', '14.484']

--- sea_level_pressure ---
[1004.1, 1002.5, 1000.5, 999.9, 999.5, 998.3, 998.5, 998.5, 998.5, 999.0]

--- precipitation ---
[0.3, 4.3, 2.2, 3.5, 6.8, 7.6, 7.8, 8.6, 7.9, 0.0]

--- ceiling_height ---
['1219', '1829', '1372', '1372', '488', '488', '274', '335', '335', '274']


In [82]:
numeric_weather_cols = [
    "temperature",
    "dew_point_temperature",
    "wind_direction",
    "wind_speed",
    "visibility",
    "sea_level_pressure",
    "precipitation",
    "ceiling_height"
]

for col in numeric_weather_cols:
    x = pd.to_numeric(sample_weather[col], errors="coerce")

    print(
        f"{col:25s}",
        f"min={x.min():10.2f}",
        f"median={x.median():10.2f}",
        f"max={x.max():10.2f}",
        f"numeric={x.notna().mean():.1%}"
    )

temperature               min=    -21.70 median=     12.20 max=     36.10 numeric=99.0%
dew_point_temperature     min=    -23.90 median=      6.70 max=     25.00 numeric=99.1%
wind_direction            min=     10.00 median=    250.00 max=    999.00 numeric=98.9%
wind_speed                min=      0.00 median=      3.10 max=     21.60 numeric=98.9%
visibility                min=      0.40 median=     16.09 max=     16.09 numeric=99.1%
sea_level_pressure        min=    984.10 median=   1016.80 max=   1036.60 numeric=82.2%
precipitation             min=      0.00 median=      0.00 max=     19.80 numeric=51.3%
ceiling_height            min=     30.00 median=   3048.00 max=  22000.00 numeric=78.6%


## Weather Transformation Layer

Raw GHCNh observations are reduced to operational weather variables relevant to flight disruption modeling.

The transformation layer:

- retains only required weather measurements;
- converts measurement fields to numeric types;
- converts timestamps to proper datetime values;
- handles special wind-direction codes;
- preserves missing observations rather than automatically interpreting them as zero;
- keeps raw NOAA files unchanged.

Selected baseline weather variables include temperature, dew point, wind speed and direction, visibility, sea-level pressure, precipitation, and ceiling height.

In [83]:
weather_glob = str(WEATHER_RAW_DIR / "GHCNh_*_2025.parquet")

weather_raw = con.sql(f"""
    SELECT
        STATION AS station_id,
        TRY_CAST(DATE AS TIMESTAMP) AS weather_datetime,

        TRY_CAST(temperature AS DOUBLE) AS temperature_c,
        TRY_CAST(dew_point_temperature AS DOUBLE) AS dew_point_c,

        CASE
            WHEN TRY_CAST(wind_direction AS DOUBLE) = 999
                THEN NULL
            ELSE TRY_CAST(wind_direction AS DOUBLE)
        END AS wind_direction_deg,

        TRY_CAST(wind_speed AS DOUBLE) AS wind_speed_ms,
        TRY_CAST(visibility AS DOUBLE) AS visibility_km,
        TRY_CAST(sea_level_pressure AS DOUBLE) AS sea_level_pressure_hpa,
        TRY_CAST(precipitation AS DOUBLE) AS precipitation_mm,
        TRY_CAST(ceiling_height AS DOUBLE) AS ceiling_height_m

    FROM read_parquet(
        '{weather_glob}',
        union_by_name = true
    )
""")

In [84]:
con.sql("""
SELECT
    COUNT(*) AS weather_rows,
    COUNT(DISTINCT station_id) AS stations,
    MIN(weather_datetime) AS first_observation,
    MAX(weather_datetime) AS last_observation
FROM weather_raw
""").df()

,weather_rows,stations,first_observation,last_observation
0,4675460,353,2025-01-01,2025-12-31 23:58:00


In [85]:
con.sql("""
SELECT
    ROUND(100.0 * COUNT(temperature_c) / COUNT(*), 2)
        AS temperature_pct,

    ROUND(100.0 * COUNT(wind_speed_ms) / COUNT(*), 2)
        AS wind_speed_pct,

    ROUND(100.0 * COUNT(visibility_km) / COUNT(*), 2)
        AS visibility_pct,

    ROUND(100.0 * COUNT(sea_level_pressure_hpa) / COUNT(*), 2)
        AS pressure_pct,

    ROUND(100.0 * COUNT(precipitation_mm) / COUNT(*), 2)
        AS precipitation_pct,

    ROUND(100.0 * COUNT(ceiling_height_m) / COUNT(*), 2)
        AS ceiling_pct

FROM weather_raw
""").df()

,temperature_pct,wind_speed_pct,visibility_pct,pressure_pct,precipitation_pct,ceiling_pct
0,94.11,93.87,91.71,70.75,47.62,79.26


In [86]:
con.sql("""
WITH hourly_counts AS (
    SELECT
        station_id,
        DATE_TRUNC('hour', weather_datetime) AS weather_hour,
        COUNT(*) AS observations
    FROM weather_raw
    GROUP BY 1, 2
)

SELECT
    observations,
    COUNT(*) AS station_hours
FROM hourly_counts
GROUP BY observations
ORDER BY observations
""").df()

,observations,station_hours
0,1,2056986
1,2,619574
2,3,170498
3,4,105480
4,5,30179
5,6,19871
6,7,5400
7,8,2180
8,9,956
9,10,396


In [87]:
weather_hourly = con.sql("""
SELECT
    station_id,
    DATE_TRUNC('hour', weather_datetime) AS weather_hour,

    AVG(temperature_c) AS temperature_c,
    AVG(dew_point_c) AS dew_point_c,

    AVG(wind_speed_ms) AS wind_speed_avg_ms,
    MAX(wind_speed_ms) AS wind_speed_max_ms,

    MIN(visibility_km) AS visibility_min_km,

    AVG(sea_level_pressure_hpa) AS sea_level_pressure_hpa,

    MAX(precipitation_mm) AS precipitation_mm,

    MIN(ceiling_height_m) AS ceiling_height_min_m,

    COUNT(*) AS observations_in_hour

FROM weather_raw

GROUP BY
    station_id,
    DATE_TRUNC('hour', weather_datetime)
""")

In [88]:
con.sql("""
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT (station_id, weather_hour)) AS unique_station_hours
FROM weather_hourly
""").df()

,rows,unique_station_hours
0,3020519,3020519


In [89]:
con.sql("""
SELECT
    COUNT(DISTINCT station_id) AS stations,
    MIN(weather_hour) AS first_hour,
    MAX(weather_hour) AS last_hour,

    ROUND(AVG(observations_in_hour), 2) AS avg_reports_per_hour,
    MAX(observations_in_hour) AS max_reports_per_hour

FROM weather_hourly
""").df()

,stations,first_hour,last_hour,avg_reports_per_hour,max_reports_per_hour
0,353,2025-01-01,2025-12-31 23:00:00,1.55,22


In [90]:
weather_features = con.sql("""
SELECT
    *,

    CASE
        WHEN visibility_min_km < 5 THEN 1
        ELSE 0
    END AS low_visibility,

    CASE
        WHEN wind_speed_max_ms >= 10 THEN 1
        ELSE 0
    END AS high_wind,

    CASE
        WHEN ceiling_height_min_m < 300 THEN 1
        ELSE 0
    END AS low_ceiling,

    CASE
        WHEN precipitation_mm > 0 THEN 1
        ELSE 0
    END AS precipitation_observed,

    CASE
        WHEN temperature_c <= 0 THEN 1
        ELSE 0
    END AS freezing_conditions

FROM weather_hourly
""")

In [91]:
weather_features = con.sql("""
SELECT
    *,

    CASE
        WHEN visibility_min_km IS NULL THEN NULL
        WHEN visibility_min_km < 5 THEN 1
        ELSE 0
    END AS low_visibility,

    CASE
        WHEN wind_speed_max_ms IS NULL THEN NULL
        WHEN wind_speed_max_ms >= 10 THEN 1
        ELSE 0
    END AS high_wind,

    CASE
        WHEN ceiling_height_min_m IS NULL THEN NULL
        WHEN ceiling_height_min_m < 300 THEN 1
        ELSE 0
    END AS low_ceiling,

    CASE
        WHEN precipitation_mm IS NULL THEN NULL
        WHEN precipitation_mm > 0 THEN 1
        ELSE 0
    END AS precipitation_observed,

    CASE
        WHEN temperature_c IS NULL THEN NULL
        WHEN temperature_c <= 0 THEN 1
        ELSE 0
    END AS freezing_conditions

FROM weather_hourly
""")

In [92]:
WEATHER_FEATURE_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "weather"
    / "ghcnh_hourly_features_2025.parquet"
)

WEATHER_FEATURE_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

con.sql(f"""
COPY (
    SELECT *
    FROM weather_features
)
TO '{WEATHER_FEATURE_PATH}'
(FORMAT PARQUET, COMPRESSION ZSTD)
""")

print(f"Saved: {WEATHER_FEATURE_PATH}")

Saved: /Users/tseringgurung/Desktop/flight-operations-intelligence/data/processed/weather/ghcnh_hourly_features_2025.parquet


In [93]:
print(
    f"Weather feature file: "
    f"{WEATHER_FEATURE_PATH.stat().st_size / 1024**2:.2f} MB"
)

Weather feature file: 25.17 MB


In [94]:
con.sql(f"""
CREATE OR REPLACE VIEW weather_features_2025 AS
SELECT *
FROM read_parquet('{WEATHER_FEATURE_PATH}')
""")

In [95]:
con.sql("""
SELECT
    COUNT(*) AS station_hours,
    COUNT(DISTINCT station_id) AS stations,
    MIN(weather_hour) AS first_hour,
    MAX(weather_hour) AS last_hour
FROM weather_features_2025
""").df()

,station_hours,stations,first_hour,last_hour
0,3020519,353,2025-01-01,2025-12-31 23:00:00


In [96]:
con.register(
    "airport_station_map_df",
    airport_station_map
)

con.sql("""
CREATE OR REPLACE VIEW airport_weather_map AS
SELECT
    airport,
    ghcnh_station_id AS station_id,
    ghcnh_station_name AS station_name,
    station_distance_miles
FROM airport_station_map_df
""")

In [97]:
con.sql("""
SELECT
    COUNT(*) AS airports,
    COUNT(DISTINCT airport) AS unique_airports,
    COUNT(DISTINCT station_id) AS unique_stations,
    MAX(station_distance_miles) AS max_distance_miles
FROM airport_weather_map
""").df()

,airports,unique_airports,unique_stations,max_distance_miles
0,352,352,352,2.83911


In [99]:
con.sql("""
SELECT
    FlightDate,
    Origin,
    Dest,
    CRSDepTime,
    DepTime,
    CRSArrTime,
    ArrTime,
    DepDelay,
    ArrDelay
FROM flights_2025
WHERE Cancelled = 0
  AND Diverted = 0
LIMIT 10
""").df()

,FlightDate,Origin,Dest,CRSDepTime,DepTime,CRSArrTime,ArrTime,DepDelay,ArrDelay
0,2025-01-01,JFK,LAX,0659,0656,1020,1013,-3.0,-7.0
1,2025-01-02,JFK,LAX,0659,0652,1020,1022,-7.0,2.0
2,2025-01-03,JFK,LAX,0659,0652,1020,1003,-7.0,-17.0
3,2025-01-04,JFK,LAX,0700,0653,1026,1016,-7.0,-10.0
4,2025-01-05,JFK,LAX,0659,0655,1020,1013,-4.0,-7.0
5,2025-01-06,JFK,LAX,0659,0703,1020,1012,4.0,-8.0
6,2025-01-07,JFK,LAX,0700,0655,1035,0955,-5.0,-40.0
7,2025-01-08,JFK,LAX,0700,0652,1035,0943,-8.0,-52.0
8,2025-01-09,JFK,LAX,0659,0654,1028,0942,-5.0,-46.0
9,2025-01-10,JFK,LAX,0700,0700,1035,1000,0.0,-35.0


In [100]:
con.sql("""
CREATE OR REPLACE VIEW flights_with_sched_ts AS

SELECT
    *,
    
    CAST(FlightDate AS DATE)
    +
    CAST(
        FLOOR(CAST(CRSDepTime AS INTEGER) / 100) AS INTEGER
    ) * INTERVAL '1 hour'
    +
    CAST(
        MOD(CAST(CRSDepTime AS INTEGER), 100) AS INTEGER
    ) * INTERVAL '1 minute'
    AS scheduled_departure_ts

FROM flights_2025
WHERE Cancelled = 0
  AND Diverted = 0
  AND ArrDelay IS NOT NULL
""")

In [101]:
con.sql("""
SELECT
    FlightDate,
    Origin,
    Dest,
    CRSDepTime,
    scheduled_departure_ts,
    ArrDelay
FROM flights_with_sched_ts
LIMIT 10
""").df()

,FlightDate,Origin,Dest,CRSDepTime,scheduled_departure_ts,ArrDelay
0,2025-01-01,JFK,LAX,0659,2025-01-01 06:59:00,-7.0
1,2025-01-02,JFK,LAX,0659,2025-01-02 06:59:00,2.0
2,2025-01-03,JFK,LAX,0659,2025-01-03 06:59:00,-17.0
3,2025-01-04,JFK,LAX,0700,2025-01-04 07:00:00,-10.0
4,2025-01-05,JFK,LAX,0659,2025-01-05 06:59:00,-7.0
5,2025-01-06,JFK,LAX,0659,2025-01-06 06:59:00,-8.0
6,2025-01-07,JFK,LAX,0700,2025-01-07 07:00:00,-40.0
7,2025-01-08,JFK,LAX,0700,2025-01-08 07:00:00,-52.0
8,2025-01-09,JFK,LAX,0659,2025-01-09 06:59:00,-46.0
9,2025-01-10,JFK,LAX,0700,2025-01-10 07:00:00,-35.0


In [102]:
jfk_station = final_airport_station_map.loc[
    final_airport_station_map["airport"] == "JFK",
    "ghcnh_station_id"
].iloc[0]

print("JFK station:", jfk_station)

JFK station: USW00094789


In [103]:
con.sql(f"""
SELECT
    station_id,
    weather_hour,
    temperature_c,
    wind_speed_avg_ms,
    visibility_min_km
FROM weather_features_2025
WHERE station_id = '{jfk_station}'
  AND CAST(weather_hour AS DATE) = DATE '2025-01-01'
ORDER BY weather_hour
LIMIT 30
""").df()

,station_id,weather_hour,temperature_c,wind_speed_avg_ms,visibility_min_km
0,USW00094789,2025-01-01 00:00:00,9.700000,7.200000,16.000
1,USW00094789,2025-01-01 01:00:00,10.600000,8.800000,16.093
2,USW00094789,2025-01-01 02:00:00,10.000000,10.466667,8.047
3,USW00094789,2025-01-01 03:00:00,10.533333,9.233333,8.000
4,USW00094789,2025-01-01 04:00:00,10.000000,3.600000,16.093
5,USW00094789,2025-01-01 05:00:00,8.900000,2.100000,16.093
6,USW00094789,2025-01-01 06:00:00,8.050000,1.800000,11.265
7,USW00094789,2025-01-01 07:00:00,8.300000,4.900000,16.093
8,USW00094789,2025-01-01 08:00:00,7.900000,4.600000,16.093
9,USW00094789,2025-01-01 09:00:00,7.800000,4.350000,16.000


In [104]:
con.sql(f"""
SELECT
    STATION,
    DATE,
    temperature,
    wind_speed,
    visibility
FROM read_parquet(
    '{WEATHER_RAW_DIR}/GHCNh_{jfk_station}_2025.parquet'
)
WHERE CAST(DATE AS DATE) = DATE '2025-01-01'
ORDER BY DATE
LIMIT 30
""").df()

,STATION,DATE,temperature,wind_speed,visibility
0,USW00094789,2025-01-01T00:00:00,9.4,7.2,16.0
1,USW00094789,2025-01-01T00:51:00,10.0,7.2,16.093
2,USW00094789,2025-01-01T01:51:00,10.6,8.8,16.093
3,USW00094789,2025-01-01T02:32:00,10.0,8.8,11.265
4,USW00094789,2025-01-01T02:49:00,10.0,10.8,8.047
5,USW00094789,2025-01-01T02:51:00,10.0,11.8,8.047
6,USW00094789,2025-01-01T03:00:00,10.0,11.8,8.0
7,USW00094789,2025-01-01T03:49:00,11.0,7.7,16.093
8,USW00094789,2025-01-01T03:51:00,10.6,8.2,16.093
9,USW00094789,2025-01-01T04:02:00,10.6,4.6,16.093


In [106]:
final_airport_station_map.columns.tolist()

['airport',
 'ARPT_NAME',
 'LAT_DECIMAL',
 'LONG_DECIMAL',
 'station_index',
 'station_distance_miles',
 'ghcnh_station_id',
 'ghcnh_station_name',
 'station_latitude',
 'station_longitude']

In [107]:
final_airport_station_map.head()

,airport,ARPT_NAME,LAT_DECIMAL,LONG_DECIMAL,station_index,station_distance_miles,ghcnh_station_id,ghcnh_station_name,station_latitude,station_longitude
0,ABE,LEHIGH VALLEY INTL,40.652363,-75.440406,32256,0.429054,USW00014737,PA ALLENTOWN LEHIGH VLY INTL A,40.6497,-75.4478
1,ABI,ABILENE RGNL,32.411333,-99.681889,32184,0.050672,USW00013962,TX ABILENE RGNL AP,32.4106,-99.6819
2,ABQ,ALBUQUERQUE INTL SUNPORT,35.038932,-106.608262,32490,0.463007,USW00023050,NM ALBUQUERQUE INTL AP,35.0419,-106.6156
3,ABR,ABERDEEN RGNL,45.446798,-98.422441,32399,0.763644,USW00014929,SD ABERDEEN,45.4558,-98.4133
4,ABY,SOUTHWEST GEORGIA RGNL,31.535530,-84.194483,32125,0.098109,USW00013869,GA ALBANY SW GEORGIA RGNL AP,31.5364,-84.1958


In [108]:
from timezonefinder import TimezoneFinder

tf = TimezoneFinder()

final_airport_station_map["timezone"] = [
    tf.timezone_at(lat=lat, lng=lon)
    for lat, lon in zip(
        final_airport_station_map["LAT_DECIMAL"],
        final_airport_station_map["LONG_DECIMAL"]
    )
]

final_airport_station_map[
    ["airport", "ARPT_NAME", "LAT_DECIMAL", "LONG_DECIMAL", "timezone"]
].head(10)

,airport,ARPT_NAME,LAT_DECIMAL,LONG_DECIMAL,timezone
0,ABE,LEHIGH VALLEY INTL,40.652363,-75.440406,America/New_York
1,ABI,ABILENE RGNL,32.411333,-99.681889,America/Chicago
2,ABQ,ALBUQUERQUE INTL SUNPORT,35.038932,-106.608262,America/Denver
3,ABR,ABERDEEN RGNL,45.446798,-98.422441,America/Chicago
4,ABY,SOUTHWEST GEORGIA RGNL,31.535530,-84.194483,America/New_York
5,ACK,NANTUCKET MEML,41.253299,-70.060511,America/New_York
6,ACT,WACO RGNL,31.612194,-97.230306,America/Chicago
7,ACV,CALIFORNIA REDWOOD COAST-HUMBOLDT COUNTY,40.977826,-124.108469,America/Los_Angeles
8,ACY,ATLANTIC CITY INTL,39.457576,-74.577155,America/New_York
9,ADK,ADAK,51.883583,-176.642482,America/Adak


In [109]:
print("Airports:", len(final_airport_station_map))
print("Missing timezones:", final_airport_station_map["timezone"].isna().sum())
print("Unique timezones:", final_airport_station_map["timezone"].nunique())

final_airport_station_map["timezone"].value_counts(dropna=False)

Airports: 352
Missing timezones: 0
Unique timezones: 22


timezone
America/Chicago                 116
America/New_York                 98
America/Denver                   39
America/Los_Angeles              37
America/Detroit                  13
America/Anchorage                 9
America/Phoenix                   6
America/Boise                     5
Pacific/Honolulu                  5
America/Sitka                     4
America/Indiana/Indianapolis      4
America/Puerto_Rico               3
America/St_Thomas                 2
America/Nome                      2
America/Juneau                    2
America/Menominee                 1
Pacific/Guam                      1
Pacific/Pago_Pago                 1
America/Kentucky/Louisville       1
Pacific/Saipan                    1
America/Adak                      1
America/Yakutat                   1
Name: count, dtype: int64

In [110]:
final_airport_station_map.loc[
    final_airport_station_map["airport"].isin(
        ["JFK", "LAX", "ORD", "DEN", "MEM", "PHX", "ANC", "HNL"]
    ),
    ["airport", "ARPT_NAME", "timezone"]
].sort_values("airport")

,airport,ARPT_NAME,timezone
17,ANC,TED STEVENS ANCHORAGE INTL,America/Anchorage
89,DEN,DENVER INTL,America/Denver
151,HNL,DANIEL K INOUYE INTL,Pacific/Honolulu
175,JFK,JOHN F KENNEDY INTL,America/New_York
187,LAX,LOS ANGELES INTL,America/Los_Angeles
212,MEM,FREDERICK W SMITH INTL/MEMPHIS,America/Chicago
242,ORD,CHICAGO O'HARE INTL,America/Chicago
254,PHX,PHOENIX SKY HARBOR INTL,America/Phoenix


## Timezone Alignment for Weather Integration

BTS scheduled departure times are recorded in each origin airport's local time, while NOAA GHCNh observation timestamps are aligned to UTC.

To create a valid point-in-time weather join, each airport is assigned an IANA timezone using its FAA latitude and longitude coordinates.

The scheduled departure timestamp is then converted from local airport time to UTC before weather matching.

This prevents systematic multi-hour weather misalignment and allows the model to use only observations that would have been available by the scheduled departure time.

In [111]:
con.unregister("airport_station_map_df")

con.register(
    "airport_station_map_df",
    final_airport_station_map
)

con.sql("""
CREATE OR REPLACE VIEW airport_weather_map AS
SELECT
    airport,
    ghcnh_station_id AS station_id,
    ghcnh_station_name AS station_name,
    station_distance_miles,
    timezone
FROM airport_station_map_df
""")

In [112]:
con.sql("""
CREATE OR REPLACE VIEW flights_weather_ready AS

SELECT
    f.*,
    m.station_id AS origin_weather_station,
    m.station_name AS origin_weather_station_name,
    m.station_distance_miles,
    m.timezone AS origin_timezone

FROM flights_with_sched_ts f

LEFT JOIN airport_weather_map m
    ON f.Origin = m.airport
""")

In [113]:
con.sql("""
SELECT
    COUNT(*) AS flights,
    SUM(CASE WHEN origin_weather_station IS NULL THEN 1 ELSE 0 END)
        AS missing_station,
    SUM(CASE WHEN origin_timezone IS NULL THEN 1 ELSE 0 END)
        AS missing_timezone
FROM flights_weather_ready
""").df()

,flights,missing_station,missing_timezone
0,6879484,0.0,0.0


In [114]:
con.sql("""
CREATE OR REPLACE VIEW flights_weather_utc AS

SELECT
    *,
    
    timezone(
        'UTC',
        timezone(
            origin_timezone,
            scheduled_departure_ts
        )
    ) AS scheduled_departure_utc

FROM flights_weather_ready
""")

In [115]:
con.sql("""
SELECT
    FlightDate,
    Origin,
    CRSDepTime,
    origin_timezone,
    scheduled_departure_ts,
    scheduled_departure_utc
FROM flights_weather_utc
WHERE Origin IN ('JFK', 'LAX', 'MEM', 'PHX', 'HNL')
ORDER BY FlightDate, Origin
LIMIT 20
""").df()

,FlightDate,Origin,CRSDepTime,origin_timezone,scheduled_departure_ts,scheduled_departure_utc
0,2025-01-01,HNL,1325,Pacific/Honolulu,2025-01-01 13:25:00,2025-01-01 23:25:00
1,2025-01-01,HNL,1435,Pacific/Honolulu,2025-01-01 14:35:00,2025-01-02 00:35:00
2,2025-01-01,HNL,1515,Pacific/Honolulu,2025-01-01 15:15:00,2025-01-02 01:15:00
3,2025-01-01,HNL,1310,Pacific/Honolulu,2025-01-01 13:10:00,2025-01-01 23:10:00
4,2025-01-01,HNL,2045,Pacific/Honolulu,2025-01-01 20:45:00,2025-01-02 06:45:00
5,2025-01-01,HNL,0600,Pacific/Honolulu,2025-01-01 06:00:00,2025-01-01 16:00:00
6,2025-01-01,HNL,0630,Pacific/Honolulu,2025-01-01 06:30:00,2025-01-01 16:30:00
7,2025-01-01,HNL,0716,Pacific/Honolulu,2025-01-01 07:16:00,2025-01-01 17:16:00
8,2025-01-01,HNL,0639,Pacific/Honolulu,2025-01-01 06:39:00,2025-01-01 16:39:00
9,2025-01-01,HNL,0805,Pacific/Honolulu,2025-01-01 08:05:00,2025-01-01 18:05:00


In [116]:
con.sql("""
SELECT
    Origin,
    origin_timezone,
    scheduled_departure_ts,
    scheduled_departure_utc
FROM flights_weather_utc
WHERE Origin = 'JFK'
  AND FlightDate = DATE '2025-01-01'
ORDER BY scheduled_departure_ts
LIMIT 10
""").df()

,Origin,origin_timezone,scheduled_departure_ts,scheduled_departure_utc
0,JFK,America/New_York,2025-01-01 05:01:00,2025-01-01 10:01:00
1,JFK,America/New_York,2025-01-01 05:01:00,2025-01-01 10:01:00
2,JFK,America/New_York,2025-01-01 05:25:00,2025-01-01 10:25:00
3,JFK,America/New_York,2025-01-01 05:45:00,2025-01-01 10:45:00
4,JFK,America/New_York,2025-01-01 06:00:00,2025-01-01 11:00:00
5,JFK,America/New_York,2025-01-01 06:00:00,2025-01-01 11:00:00
6,JFK,America/New_York,2025-01-01 06:00:00,2025-01-01 11:00:00
7,JFK,America/New_York,2025-01-01 06:00:00,2025-01-01 11:00:00
8,JFK,America/New_York,2025-01-01 06:00:00,2025-01-01 11:00:00
9,JFK,America/New_York,2025-01-01 06:00:00,2025-01-01 11:00:00


In [119]:
from pathlib import Path

weather_dir = Path("../data/raw/weather/ghcnh/2025")

print("Exists:", weather_dir.exists())
print("Parquet files:", len(list(weather_dir.glob("*.parquet"))))

Exists: True
Parquet files: 353


In [120]:
con.sql("""
CREATE OR REPLACE VIEW weather_observations AS

SELECT
    STATION AS station_id,
    CAST(DATE AS TIMESTAMP) AS observation_ts,
    TRY_CAST(temperature AS DOUBLE) AS temperature_c,
    TRY_CAST(wind_speed AS DOUBLE) AS wind_speed_ms,
    TRY_CAST(visibility AS DOUBLE) AS visibility_km,
    TRY_CAST(sea_level_pressure AS DOUBLE) AS sea_level_pressure,
    TRY_CAST(precipitation AS DOUBLE) AS precipitation,
    TRY_CAST(ceiling_height AS DOUBLE) AS ceiling_height

FROM read_parquet(
    '../data/raw/weather/ghcnh/2025/*.parquet',
    union_by_name = true
)
""")

In [121]:
con.sql("""
SELECT
    station_id,
    observation_ts,
    temperature_c,
    wind_speed_ms,
    visibility_km,
    sea_level_pressure,
    precipitation,
    ceiling_height
FROM weather_observations
ORDER BY station_id, observation_ts
LIMIT 20
""").df()

,station_id,observation_ts,temperature_c,wind_speed_ms,visibility_km,sea_level_pressure,precipitation,ceiling_height
0,ACW00011647,2025-01-01 00:00:00,24.6,2.1,30.0,1014.4,NaN,8400.0
1,ACW00011647,2025-01-01 01:00:00,24.4,2.1,30.0,1014.7,NaN,8400.0
2,ACW00011647,2025-01-01 02:00:00,24.4,1.0,30.0,1014.8,NaN,8400.0
3,ACW00011647,2025-01-01 03:00:00,24.4,0.5,30.0,1014.8,NaN,8400.0
4,ACW00011647,2025-01-01 04:00:00,24.8,2.6,30.0,1014.5,NaN,8400.0
5,ACW00011647,2025-01-01 05:00:00,24.9,3.1,28.0,1014.3,NaN,8400.0
6,ACW00011647,2025-01-01 06:00:00,24.4,2.6,30.0,1013.9,NaN,8400.0
7,ACW00011647,2025-01-01 07:00:00,23.8,2.6,30.0,1013.4,NaN,8400.0
8,ACW00011647,2025-01-01 08:00:00,24.6,2.6,30.0,1013.4,NaN,540.0
9,ACW00011647,2025-01-01 09:00:00,23.3,2.6,28.0,1013.8,NaN,1080.0


In [122]:
con.sql("""
SELECT
    COUNT(*) AS observations,
    COUNT(DISTINCT station_id) AS stations,
    MIN(observation_ts) AS first_observation,
    MAX(observation_ts) AS last_observation
FROM weather_observations
""").df()

,observations,stations,first_observation,last_observation
0,4675460,353,2025-01-01,2025-12-31 23:58:00


### Restrict Weather Data to Final Selected Stations

The raw weather directory contains 353 station files because one additional NOAA station file was downloaded during URL validation.

Only the 352 stations included in the finalized airport-to-station crosswalk are retained for modeling. This prevents test/download artifacts from entering the production weather feature layer.

In [123]:
selected_stations_df = (
    final_airport_station_map[
        ["ghcnh_station_id"]
    ]
    .drop_duplicates()
    .rename(columns={"ghcnh_station_id": "station_id"})
)

con.register(
    "selected_stations_df",
    selected_stations_df
)

In [124]:
con.sql("""
CREATE OR REPLACE VIEW weather_observations_selected AS

SELECT
    w.*
FROM weather_observations w

INNER JOIN selected_stations_df s
    ON w.station_id = s.station_id
""")

In [125]:
con.sql("""
SELECT
    COUNT(*) AS observations,
    COUNT(DISTINCT station_id) AS stations,
    MIN(observation_ts) AS first_observation,
    MAX(observation_ts) AS last_observation
FROM weather_observations_selected
""").df()

,observations,stations,first_observation,last_observation
0,4666637,352,2025-01-01,2025-12-31 23:58:00


In [126]:
con.sql("""
CREATE OR REPLACE VIEW weather_hourly_selected AS

SELECT
    station_id,
    DATE_TRUNC('hour', observation_ts) AS weather_hour,

    AVG(temperature_c) AS temperature_c,
    AVG(wind_speed_ms) AS wind_speed_avg_ms,
    MAX(wind_speed_ms) AS wind_speed_max_ms,
    MIN(visibility_km) AS visibility_min_km,
    AVG(sea_level_pressure) AS pressure_avg,
    MAX(precipitation) AS precipitation_max,
    MIN(ceiling_height) AS ceiling_min

FROM weather_observations_selected

GROUP BY
    station_id,
    DATE_TRUNC('hour', observation_ts)
""")

In [127]:
con.sql("""
SELECT
    COUNT(*) AS station_hours,
    COUNT(DISTINCT station_id) AS stations,
    MIN(weather_hour) AS first_hour,
    MAX(weather_hour) AS last_hour
FROM weather_hourly_selected
""").df()

,station_hours,stations,first_hour,last_hour
0,3011928,352,2025-01-01,2025-12-31 23:00:00


In [128]:
con.sql("""
SELECT *
FROM weather_hourly_selected
WHERE station_id = (
    SELECT ghcnh_station_id
    FROM final_airport_station_map
    WHERE airport = 'JFK'
)
AND weather_hour >= TIMESTAMP '2025-01-01 08:00:00'
AND weather_hour <= TIMESTAMP '2025-01-01 14:00:00'
ORDER BY weather_hour
""").df()

,station_id,weather_hour,temperature_c,wind_speed_avg_ms,wind_speed_max_ms,visibility_min_km,pressure_avg,precipitation_max,ceiling_min
0,USW00094789,2025-01-01 08:00:00,7.90,4.600000,4.6,16.093,998.30,0.0,3962.0
1,USW00094789,2025-01-01 09:00:00,7.80,4.350000,4.6,16.000,998.55,0.0,3353.0
2,USW00094789,2025-01-01 10:00:00,8.30,5.700000,5.7,16.093,998.90,0.0,3353.0
3,USW00094789,2025-01-01 11:00:00,8.15,5.700000,5.7,12.875,999.20,0.0,1829.0
4,USW00094789,2025-01-01 12:00:00,8.50,5.666667,6.2,12.000,999.20,0.0,305.0
5,USW00094789,2025-01-01 13:00:00,9.40,6.200000,6.2,16.093,999.80,0.0,1981.0
6,USW00094789,2025-01-01 14:00:00,9.15,6.450000,6.7,16.093,1000.30,0.0,366.0


In [129]:
con.sql("""
SELECT
    STATION,
    DATE,
    temperature,
    wind_speed,
    visibility
FROM read_parquet(
    '../data/raw/weather/ghcnh/2025/*.parquet',
    union_by_name = true
)
WHERE STATION = 'USW00094789'
  AND CAST(DATE AS TIMESTAMP)
      BETWEEN TIMESTAMP '2025-01-01 09:30:00'
          AND TIMESTAMP '2025-01-01 11:30:00'
ORDER BY DATE
""").df()

,STATION,DATE,temperature,wind_speed,visibility
0,USW00094789,2025-01-01T09:51:00,7.8,4.1,16.093
1,USW00094789,2025-01-01T10:51:00,8.3,5.7,16.093


In [131]:
con.sql("""
CREATE OR REPLACE VIEW flights_weather_base AS

SELECT
    f.*,
    m.ghcnh_station_id AS origin_station_id

FROM flights_weather_utc f


LEFT JOIN final_airport_station_map m
    ON f.Origin = m.airport
""")

In [132]:
con.sql("""
SELECT
    COUNT(*) AS flights,
    SUM(origin_station_id IS NULL) AS missing_station
FROM flights_weather_base
""").df()

,flights,missing_station
0,6879484,0.0


In [133]:
con.sql("""
CREATE OR REPLACE TABLE flights_with_weather AS

SELECT
    f.*,

    w.observation_ts AS weather_observation_ts,
    w.temperature_c,
    w.wind_speed_ms,
    w.visibility_km,
    w.sea_level_pressure,
    w.precipitation,
    w.ceiling_height,

    DATE_DIFF(
        'minute',
        w.observation_ts,
        f.scheduled_departure_utc
    ) AS weather_age_minutes

FROM flights_weather_base f

ASOF LEFT JOIN weather_observations_selected w
    ON f.origin_station_id = w.station_id
   AND f.scheduled_departure_utc >= w.observation_ts
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [134]:
con.sql("""
SELECT
    COUNT(*) AS flights,
    COUNT(weather_observation_ts) AS weather_matched,
    COUNT(*) - COUNT(weather_observation_ts) AS weather_missing,

    ROUND(
        100.0 * COUNT(weather_observation_ts) / COUNT(*),
        2
    ) AS match_rate_pct,

    MIN(weather_age_minutes) AS min_age_minutes,
    MEDIAN(weather_age_minutes) AS median_age_minutes,
    MAX(weather_age_minutes) AS max_age_minutes

FROM flights_with_weather
""").df()

,flights,weather_matched,weather_missing,match_rate_pct,min_age_minutes,median_age_minutes,max_age_minutes
0,6879484,6879481,3,100.0,0,24.0,182363


### Weather Match Staleness Validation

The point-in-time ASOF join successfully matched nearly every flight to the most recent weather observation available before scheduled departure.

However, a successful match does not guarantee that the observation is sufficiently recent to represent current airport conditions. Because an ASOF join will continue searching backward until it finds a prior observation, stations with reporting gaps can produce extremely stale weather matches.

Weather observation age is therefore explicitly evaluated before features are accepted for modeling. Flights whose most recent observation exceeds an operational staleness threshold will be treated as having unavailable weather rather than carrying old conditions forward.

In [135]:
con.sql("""
SELECT
    MIN(weather_age_minutes) AS min_age,
    QUANTILE_CONT(weather_age_minutes, 0.50) AS median_age,
    QUANTILE_CONT(weather_age_minutes, 0.75) AS p75,
    QUANTILE_CONT(weather_age_minutes, 0.90) AS p90,
    QUANTILE_CONT(weather_age_minutes, 0.95) AS p95,
    QUANTILE_CONT(weather_age_minutes, 0.99) AS p99,
    QUANTILE_CONT(weather_age_minutes, 0.999) AS p999,
    MAX(weather_age_minutes) AS max_age
FROM flights_with_weather
WHERE weather_observation_ts IS NOT NULL
""").df()

,min_age,median_age,p75,p90,p95,p99,p999,max_age
0,0,24.0,40.0,51.0,56.0,169.0,5126.52,182363


In [136]:
con.sql("""
SELECT
    CASE
        WHEN weather_age_minutes <= 60 THEN '<= 60 min'
        WHEN weather_age_minutes <= 120 THEN '61-120 min'
        WHEN weather_age_minutes <= 180 THEN '121-180 min'
        WHEN weather_age_minutes <= 360 THEN '181-360 min'
        WHEN weather_age_minutes <= 1440 THEN '6-24 hr'
        ELSE '> 24 hr'
    END AS age_band,

    COUNT(*) AS flights,

    ROUND(
        COUNT(*) * 100.0 /
        SUM(COUNT(*)) OVER (),
        4
    ) AS pct

FROM flights_with_weather
WHERE weather_observation_ts IS NOT NULL

GROUP BY 1

ORDER BY
    MIN(weather_age_minutes)
""").df()

,age_band,flights,pct
0,<= 60 min,6792429,98.7346
1,61-120 min,13474,0.1959
2,121-180 min,5618,0.0817
3,181-360 min,3418,0.0497
4,6-24 hr,16494,0.2398
5,> 24 hr,48048,0.6984


In [137]:
con.sql("""
SELECT
    FlightDate,
    Origin,
    origin_station_id,
    scheduled_departure_utc,
    weather_observation_ts,
    weather_age_minutes
FROM flights_with_weather
ORDER BY weather_age_minutes DESC
LIMIT 25
""").df()

,FlightDate,Origin,origin_station_id,scheduled_departure_utc,weather_observation_ts,weather_age_minutes
0,2025-12-31,RIW,USW00024061,2025-12-31 21:16:00,2025-08-27 05:53:00,182363
1,2025-12-31,RIW,USW00024061,2025-12-31 12:26:00,2025-08-27 05:53:00,181833
2,2025-12-31,SPN,CQW00041418,2025-12-30 20:00:00,2025-08-26 13:52:00,181808
3,2025-12-30,RIW,USW00024061,2025-12-30 21:16:00,2025-08-27 05:53:00,180923
4,2025-12-30,SPN,CQW00041418,2025-12-29 23:50:00,2025-08-26 13:52:00,180598
5,2025-12-30,RIW,USW00024061,2025-12-30 12:26:00,2025-08-27 05:53:00,180393
6,2025-12-29,RIW,USW00024061,2025-12-29 21:16:00,2025-08-27 05:53:00,179483
7,2025-12-29,RIW,USW00024061,2025-12-29 12:26:00,2025-08-27 05:53:00,178953
8,2025-12-29,SPN,CQW00041418,2025-12-28 20:00:00,2025-08-26 13:52:00,178928
9,2025-12-28,RIW,USW00024061,2025-12-28 21:16:00,2025-08-27 05:53:00,178043


In [138]:
con.sql("""
CREATE OR REPLACE TABLE flights_weather_clean AS

SELECT
    * EXCLUDE (
        temperature_c,
        wind_speed_ms,
        visibility_km,
        sea_level_pressure,
        precipitation,
        ceiling_height
    ),

    CASE WHEN weather_age_minutes <= 120
         THEN temperature_c END AS temperature_c,

    CASE WHEN weather_age_minutes <= 120
         THEN wind_speed_ms END AS wind_speed_ms,

    CASE WHEN weather_age_minutes <= 120
         THEN visibility_km END AS visibility_km,

    CASE WHEN weather_age_minutes <= 120
         THEN sea_level_pressure END AS sea_level_pressure,

    CASE WHEN weather_age_minutes <= 120
         THEN precipitation END AS precipitation,

    CASE WHEN weather_age_minutes <= 120
         THEN ceiling_height END AS ceiling_height,

    CASE
        WHEN weather_observation_ts IS NULL THEN 0
        WHEN weather_age_minutes <= 120 THEN 1
        ELSE 0
    END AS valid_weather_match

FROM flights_with_weather
""")

In [139]:
con.sql("""
SELECT
    COUNT(*) AS flights,

    SUM(valid_weather_match) AS valid_weather_matches,

    COUNT(*) - SUM(valid_weather_match) AS invalid_or_missing,

    ROUND(
        100.0 * SUM(valid_weather_match) / COUNT(*),
        2
    ) AS valid_weather_pct

FROM flights_weather_clean
""").df()

,flights,valid_weather_matches,invalid_or_missing,valid_weather_pct
0,6879484,6805903.0,73581.0,98.93


In [140]:
con.sql("""
SELECT
    SUM(
        CASE
            WHEN valid_weather_match = 0
             AND (
                 temperature_c IS NOT NULL OR
                 wind_speed_ms IS NOT NULL OR
                 visibility_km IS NOT NULL OR
                 sea_level_pressure IS NOT NULL OR
                 precipitation IS NOT NULL OR
                 ceiling_height IS NOT NULL
             )
            THEN 1
            ELSE 0
        END
    ) AS stale_weather_leakage
FROM flights_weather_clean
""").df()

,stale_weather_leakage
0,0.0


### Weather Feature Engineering

Raw NOAA observations are transformed into modeling-ready operational features.

Continuous measurements are retained to avoid imposing arbitrary weather thresholds before exploratory analysis. Additional missingness indicators are created because unavailable weather measurements may themselves contain information about station reporting conditions.

All weather features are derived from observations available no later than 120 minutes before the scheduled departure time.

In [141]:
con.sql("""
CREATE OR REPLACE VIEW flights_weather_features AS

SELECT
    *,

    -- Continuous weather features
    temperature_c AS wx_temperature_c,
    wind_speed_ms AS wx_wind_speed_ms,
    visibility_km AS wx_visibility_km,
    sea_level_pressure AS wx_pressure_hpa,
    precipitation AS wx_precipitation,
    ceiling_height AS wx_ceiling_m,

    -- Missingness indicators
    CASE WHEN temperature_c IS NULL THEN 1 ELSE 0 END
        AS wx_temperature_missing,

    CASE WHEN wind_speed_ms IS NULL THEN 1 ELSE 0 END
        AS wx_wind_missing,

    CASE WHEN visibility_km IS NULL THEN 1 ELSE 0 END
        AS wx_visibility_missing,

    CASE WHEN sea_level_pressure IS NULL THEN 1 ELSE 0 END
        AS wx_pressure_missing,

    CASE WHEN precipitation IS NULL THEN 1 ELSE 0 END
        AS wx_precipitation_missing,

    CASE WHEN ceiling_height IS NULL THEN 1 ELSE 0 END
        AS wx_ceiling_missing

FROM flights_weather_clean
""")

In [142]:
con.sql("""
SELECT
    COUNT(*) AS flights,

    ROUND(100.0 * COUNT(wx_temperature_c) / COUNT(*), 2)
        AS temperature_pct,

    ROUND(100.0 * COUNT(wx_wind_speed_ms) / COUNT(*), 2)
        AS wind_pct,

    ROUND(100.0 * COUNT(wx_visibility_km) / COUNT(*), 2)
        AS visibility_pct,

    ROUND(100.0 * COUNT(wx_pressure_hpa) / COUNT(*), 2)
        AS pressure_pct,

    ROUND(100.0 * COUNT(wx_precipitation) / COUNT(*), 2)
        AS precipitation_pct,

    ROUND(100.0 * COUNT(wx_ceiling_m) / COUNT(*), 2)
        AS ceiling_pct

FROM flights_weather_features
""").df()

,flights,temperature_pct,wind_pct,visibility_pct,pressure_pct,precipitation_pct,ceiling_pct
0,6879484,98.39,98.29,98.37,91.84,48.41,75.24


In [144]:
con.sql("""
SELECT
    CASE
        WHEN ArrDelay >= 15 THEN 1
        ELSE 0
    END AS significant_arrival_delay,

    COUNT(*) AS flights,

    ROUND(AVG(wx_temperature_c), 2)
        AS avg_temperature,

    ROUND(AVG(wx_wind_speed_ms), 2)
        AS avg_wind_speed,

    ROUND(AVG(wx_visibility_km), 2)
        AS avg_visibility,

    ROUND(AVG(wx_pressure_hpa), 2)
        AS avg_pressure,

    ROUND(AVG(wx_precipitation), 2)
        AS avg_precipitation,

    ROUND(AVG(wx_ceiling_m), 2)
        AS avg_ceiling

FROM flights_weather_features

GROUP BY
    CASE
        WHEN ArrDelay >= 15 THEN 1
        ELSE 0
    END

ORDER BY significant_arrival_delay
""").df()

,significant_arrival_delay,flights,avg_temperature,avg_wind_speed,avg_visibility,avg_pressure,avg_precipitation,avg_ceiling
0,0,5344846,17.35,3.83,15.32,1016.71,0.07,13192.4
1,1,1534638,18.11,4.30,14.85,1015.92,0.27,11537.8


In [145]:
con.sql("""
SELECT
    CASE
        WHEN wx_precipitation = 0 THEN '0 - None'
        WHEN wx_precipitation <= 1 THEN '1 - <=1'
        WHEN wx_precipitation <= 5 THEN '2 - 1-5'
        WHEN wx_precipitation <= 10 THEN '3 - 5-10'
        ELSE '4 - >10'
    END AS precipitation_band,

    COUNT(*) AS flights,

    ROUND(
        100.0 * AVG(
            CASE WHEN ArrDelay >= 15 THEN 1 ELSE 0 END
        ),
        2
    ) AS delay_rate_pct

FROM flights_weather_features

WHERE valid_weather_match = 1
  AND wx_precipitation IS NOT NULL

GROUP BY 1
ORDER BY 1
""").df()

,precipitation_band,flights,delay_rate_pct
0,0 - None,3107472,22.60
1,1 - <=1,142707,36.48
2,2 - 1-5,63848,41.79
3,3 - 5-10,9958,57.92
4,4 - >10,6305,75.62


In [146]:
con.sql("""
SELECT
    CASE
        WHEN wx_visibility_km < 1 THEN '1 - <1 km'
        WHEN wx_visibility_km < 5 THEN '2 - 1-5 km'
        WHEN wx_visibility_km < 10 THEN '3 - 5-10 km'
        ELSE '4 - >=10 km'
    END AS visibility_band,

    COUNT(*) AS flights,

    ROUND(
        100.0 * AVG(
            CASE WHEN ArrDelay >= 15 THEN 1 ELSE 0 END
        ),
        2
    ) AS delay_rate_pct

FROM flights_weather_features

WHERE valid_weather_match = 1
  AND wx_visibility_km IS NOT NULL

GROUP BY 1
ORDER BY 1
""").df()

,visibility_band,flights,delay_rate_pct
0,1 - <1 km,41622,33.08
1,2 - 1-5 km,177307,37.62
2,3 - 5-10 km,205121,29.05
3,4 - >=10 km,6343238,21.66


In [147]:
con.sql("""
SELECT
    CASE
        WHEN wx_wind_speed_ms < 5 THEN '1 - <5 m/s'
        WHEN wx_wind_speed_ms < 10 THEN '2 - 5-10 m/s'
        WHEN wx_wind_speed_ms < 15 THEN '3 - 10-15 m/s'
        ELSE '4 - >=15 m/s'
    END AS wind_band,

    COUNT(*) AS flights,

    ROUND(
        100.0 * AVG(
            CASE WHEN ArrDelay >= 15 THEN 1 ELSE 0 END
        ),
        2
    ) AS delay_rate_pct

FROM flights_weather_features

WHERE valid_weather_match = 1
  AND wx_wind_speed_ms IS NOT NULL

GROUP BY 1
ORDER BY 1
""").df()

,wind_band,flights,delay_rate_pct
0,1 - <5 m/s,4698858,20.76
1,2 - 5-10 m/s,1940427,25.61
2,3 - 10-15 m/s,119549,32.23
3,4 - >=15 m/s,3063,49.20


In [148]:
con.sql("""
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT CONCAT(
        FlightDate, '|',
        Reporting_Airline, '|',
        Flight_Number_Reporting_Airline, '|',
        Origin, '|',
        Dest, '|',
        CRSDepTime
    )) AS approximate_unique_flights,

    COUNT(*) - COUNT(DISTINCT CONCAT(
        FlightDate, '|',
        Reporting_Airline, '|',
        Flight_Number_Reporting_Airline, '|',
        Origin, '|',
        Dest, '|',
        CRSDepTime
    )) AS possible_duplicates

FROM flights_weather_features
""").df()

,rows,approximate_unique_flights,possible_duplicates
0,6879484,6879484,0


In [149]:
con.sql("""
DESCRIBE flights_weather_features
""").df()

,column_name,column_type,null,key,default,extra
0,Year,BIGINT,YES,None,None,None
1,Quarter,BIGINT,YES,None,None,None
2,Month,BIGINT,YES,None,None,None
3,DayofMonth,BIGINT,YES,None,None,None
4,DayOfWeek,BIGINT,YES,None,None,None
...,...,...,...,...,...,...
133,wx_wind_missing,INTEGER,YES,None,None,None
134,wx_visibility_missing,INTEGER,YES,None,None,None
135,wx_pressure_missing,INTEGER,YES,None,None,None
136,wx_precipitation_missing,INTEGER,YES,None,None,None


In [150]:
cols = con.sql("""
DESCRIBE flights_weather_features
""").df()

for i, col in enumerate(cols["column_name"]):
    print(f"{i}: {col}")

0: Year
1: Quarter
2: Month
3: DayofMonth
4: DayOfWeek
5: FlightDate
6: Reporting_Airline
7: DOT_ID_Reporting_Airline
8: IATA_CODE_Reporting_Airline
9: Tail_Number
10: Flight_Number_Reporting_Airline
11: OriginAirportID
12: OriginAirportSeqID
13: OriginCityMarketID
14: Origin
15: OriginCityName
16: OriginState
17: OriginStateFips
18: OriginStateName
19: OriginWac
20: DestAirportID
21: DestAirportSeqID
22: DestCityMarketID
23: Dest
24: DestCityName
25: DestState
26: DestStateFips
27: DestStateName
28: DestWac
29: CRSDepTime
30: DepTime
31: DepDelay
32: DepDelayMinutes
33: DepDel15
34: DepartureDelayGroups
35: DepTimeBlk
36: TaxiOut
37: WheelsOff
38: WheelsOn
39: TaxiIn
40: CRSArrTime
41: ArrTime
42: ArrDelay
43: ArrDelayMinutes
44: ArrDel15
45: ArrivalDelayGroups
46: ArrTimeBlk
47: Cancelled
48: CancellationCode
49: Diverted
50: CRSElapsedTime
51: ActualElapsedTime
52: AirTime
53: Flights
54: Distance
55: DistanceGroup
56: CarrierDelay
57: WeatherDelay
58: NASDelay
59: SecurityDelay
60:

In [151]:
for start in range(25, 134, 25):
    end = min(start + 25, 134)

    print(f"\n--- COLUMNS {start}–{end-1} ---")

    for i in range(start, end):
        print(f"{i}: {cols.loc[i, 'column_name']}")


--- COLUMNS 25–49 ---
25: DestState
26: DestStateFips
27: DestStateName
28: DestWac
29: CRSDepTime
30: DepTime
31: DepDelay
32: DepDelayMinutes
33: DepDel15
34: DepartureDelayGroups
35: DepTimeBlk
36: TaxiOut
37: WheelsOff
38: WheelsOn
39: TaxiIn
40: CRSArrTime
41: ArrTime
42: ArrDelay
43: ArrDelayMinutes
44: ArrDel15
45: ArrivalDelayGroups
46: ArrTimeBlk
47: Cancelled
48: CancellationCode
49: Diverted

--- COLUMNS 50–74 ---
50: CRSElapsedTime
51: ActualElapsedTime
52: AirTime
53: Flights
54: Distance
55: DistanceGroup
56: CarrierDelay
57: WeatherDelay
58: NASDelay
59: SecurityDelay
60: LateAircraftDelay
61: FirstDepTime
62: TotalAddGTime
63: LongestAddGTime
64: DivAirportLandings
65: DivReachedDest
66: DivActualElapsedTime
67: DivArrDelay
68: DivDistance
69: Div1Airport
70: Div1AirportID
71: Div1AirportSeqID
72: Div1WheelsOn
73: Div1TotalGTime
74: Div1LongestGTime

--- COLUMNS 75–99 ---
75: Div1WheelsOff
76: Div1TailNum
77: Div2Airport
78: Div2AirportID
79: Div2AirportSeqID
80: Div2W

In [153]:
con.sql("""
CREATE OR REPLACE VIEW ml_ready AS

SELECT
    -- =========================
    -- TARGET
    -- =========================
    CASE
        WHEN ArrDelay >= 15 THEN 1
        ELSE 0
    END AS significant_arrival_delay,

    -- =========================
    -- CALENDAR / SCHEDULE
    -- =========================
    Year,
    Quarter,
    Month,
    DayofMonth,
    DayOfWeek,

    EXTRACT(HOUR FROM scheduled_departure_ts) AS scheduled_dep_hour,
CAST(FLOOR(TRY_CAST(CRSArrTime AS DOUBLE) / 100) AS INTEGER) AS scheduled_arr_hour,

    -- =========================
    -- FLIGHT / NETWORK
    -- =========================
    Reporting_Airline,
    Origin,
    Dest,

    Origin || '_' || Dest AS route,

    Distance,
    CRSElapsedTime,

    -- =========================
    -- ORIGIN WEATHER
    -- =========================
    wx_temperature_c,
    wx_wind_speed_ms,
    wx_visibility_km,
    wx_pressure_hpa,
    wx_precipitation,
    wx_ceiling_m,

    -- Missingness indicators
    wx_temperature_missing,
    wx_wind_missing,
    wx_visibility_missing,
    wx_pressure_missing,
    wx_precipitation_missing,
    wx_ceiling_missing

FROM flights_weather_features

WHERE Cancelled = 0
  AND Diverted = 0
  AND ArrDelay IS NOT NULL
""")

In [154]:
con.sql("""
SELECT
    COUNT(*) AS rows,
    SUM(significant_arrival_delay) AS delayed,
    ROUND(
        100.0 * AVG(significant_arrival_delay),
        2
    ) AS delay_rate_pct
FROM ml_ready
""").df()

,rows,delayed,delay_rate_pct
0,6879484,1534638.0,22.31


In [155]:
con.sql("""
DESCRIBE ml_ready
""").df()

,column_name,column_type,null,key,default,extra
0,significant_arrival_delay,INTEGER,YES,None,None,None
1,Year,BIGINT,YES,None,None,None
2,Quarter,BIGINT,YES,None,None,None
3,Month,BIGINT,YES,None,None,None
4,DayofMonth,BIGINT,YES,None,None,None
5,DayOfWeek,BIGINT,YES,None,None,None
6,scheduled_dep_hour,BIGINT,YES,None,None,None
7,scheduled_arr_hour,INTEGER,YES,None,None,None
8,Reporting_Airline,VARCHAR,YES,None,None,None
9,Origin,VARCHAR,YES,None,None,None


In [156]:
con.sql("""
SELECT
    CRSDepTime,
    typeof(CRSDepTime) AS dep_type,
    CRSArrTime,
    typeof(CRSArrTime) AS arr_type,
    scheduled_departure_ts,
    typeof(scheduled_departure_ts) AS ts_type
FROM flights_weather_features
LIMIT 5
""").df()

,CRSDepTime,dep_type,CRSArrTime,arr_type,scheduled_departure_ts,ts_type
0,0705,VARCHAR,0820,VARCHAR,2025-01-01 07:05:00,TIMESTAMP
1,1058,VARCHAR,1208,VARCHAR,2025-01-01 10:58:00,TIMESTAMP
2,1609,VARCHAR,1810,VARCHAR,2025-01-01 16:09:00,TIMESTAMP
3,1830,VARCHAR,1940,VARCHAR,2025-01-01 18:30:00,TIMESTAMP
4,0500,VARCHAR,0615,VARCHAR,2025-01-02 05:00:00,TIMESTAMP


## Weather Modeling Base

A leakage-safe modeling base is constructed from scheduled flight information and point-in-time NOAA weather observations.

BTS scheduled departure and arrival times are stored as four-digit character fields. Scheduled departure hour is derived from the previously validated departure timestamp, while scheduled arrival hour is parsed explicitly from the BTS HHMM representation.

Post-departure and outcome variables—including actual departure/arrival times, departure delays, arrival-delay components, taxi times, and diversion information—are excluded from the predictor set.

This weather modeling base will subsequently be combined with the previously validated Baseline v2 historical features so that the incremental predictive value of NOAA weather can be evaluated fairly.

In [157]:
con.sql("""
CREATE OR REPLACE VIEW weather_model_base AS

SELECT
    -- =========================
    -- TARGET
    -- =========================
    CASE
        WHEN ArrDelay >= 15 THEN 1
        ELSE 0
    END AS significant_arrival_delay,

    -- Keep date for temporal splitting
    FlightDate,

    -- =========================
    -- CALENDAR / SCHEDULE
    -- =========================
    Year,
    Quarter,
    Month,
    DayofMonth,
    DayOfWeek,

    EXTRACT(
        HOUR FROM scheduled_departure_ts
    )::INTEGER AS scheduled_dep_hour,

    CAST(
        FLOOR(
            TRY_CAST(CRSArrTime AS DOUBLE) / 100
        )
        AS INTEGER
    ) AS scheduled_arr_hour,

    CASE
        WHEN DayOfWeek IN (6, 7) THEN 1
        ELSE 0
    END AS is_weekend,

    -- =========================
    -- FLIGHT / NETWORK
    -- =========================
    Reporting_Airline,
    Origin,
    Dest,

    Origin || '_' || Dest AS route,

    CRSElapsedTime,
    Distance,
    DistanceGroup,

    -- =========================
    -- NOAA WEATHER
    -- =========================
    wx_temperature_c,
    wx_wind_speed_ms,
    wx_visibility_km,
    wx_pressure_hpa,
    wx_precipitation,
    wx_ceiling_m,

    -- Weather missingness indicators
    wx_temperature_missing,
    wx_wind_missing,
    wx_visibility_missing,
    wx_pressure_missing,
    wx_precipitation_missing,
    wx_ceiling_missing

FROM flights_weather_features

WHERE Cancelled = 0
  AND Diverted = 0
  AND ArrDelay IS NOT NULL
""")

In [158]:
con.sql("""
SELECT
    COUNT(*) AS rows,
    MIN(FlightDate) AS first_date,
    MAX(FlightDate) AS last_date,

    SUM(significant_arrival_delay) AS delayed_flights,

    ROUND(
        AVG(significant_arrival_delay) * 100,
        2
    ) AS delay_rate_pct

FROM weather_model_base
""").df()

,rows,first_date,last_date,delayed_flights,delay_rate_pct
0,6879484,2025-01-01,2025-12-31,1534638.0,22.31


In [159]:
con.sql("""
SELECT
    FlightDate,
    CRSDepTime,
    scheduled_dep_hour,
    CRSArrTime,
    scheduled_arr_hour
FROM flights_weather_features f
JOIN weather_model_base m
USING (FlightDate, Reporting_Airline, Origin, Dest)
LIMIT 10
""").df()

,FlightDate,CRSDepTime,scheduled_dep_hour,CRSArrTime,scheduled_arr_hour
0,2025-12-28,0600,19,0728,20
1,2025-12-28,0600,17,1008,21
2,2025-12-28,0600,18,0900,21
3,2025-12-28,0615,20,1045,0
4,2025-12-28,0627,18,0959,21
5,2025-12-28,0627,19,0916,22
6,2025-12-28,0629,6,0910,9
7,2025-12-28,0630,21,0917,0
8,2025-12-28,0630,18,1056,22
9,2025-12-28,0640,5,0818,6


In [160]:
con.sql("""
SELECT
    FlightDate,
    scheduled_dep_hour,
    scheduled_arr_hour
FROM weather_model_base
LIMIT 10
""").df()

,FlightDate,scheduled_dep_hour,scheduled_arr_hour
0,2025-01-01,7,8
1,2025-01-01,10,12
2,2025-01-01,16,18
3,2025-01-01,18,19
4,2025-01-02,5,6
5,2025-01-02,7,8
6,2025-01-02,14,15
7,2025-01-02,16,18
8,2025-01-02,18,19
9,2025-01-03,5,6


## Persist Flight-Level Weather Features

The NOAA integration pipeline has now produced leakage-safe, point-in-time weather features aligned to each flight's scheduled departure.

Before returning to model development, the required weather features are persisted as a compact flight-level artifact.

A composite flight key consisting of flight date, reporting airline, flight number, origin, destination, and scheduled departure time is retained so that weather features can be joined back to the previously validated Baseline v2 modeling table without rebuilding the weather pipeline.

This separation keeps weather ingestion and model development independently reproducible.

In [161]:
con.sql("""
CREATE OR REPLACE VIEW weather_ml_export AS

SELECT
    -- Flight join key
    FlightDate,
    Reporting_Airline,
    Flight_Number_Reporting_Airline,
    Origin,
    Dest,
    CRSDepTime,

    -- Point-in-time weather features
    wx_temperature_c,
    wx_wind_speed_ms,
    wx_visibility_km,
    wx_pressure_hpa,
    wx_precipitation,
    wx_ceiling_m,

    -- Missingness indicators
    wx_temperature_missing,
    wx_wind_missing,
    wx_visibility_missing,
    wx_pressure_missing,
    wx_precipitation_missing,
    wx_ceiling_missing

FROM flights_weather_features
""")

In [162]:
con.sql("""
SELECT
    COUNT(*) AS rows,

    COUNT(
        DISTINCT CONCAT(
            FlightDate, '|',
            Reporting_Airline, '|',
            Flight_Number_Reporting_Airline, '|',
            Origin, '|',
            Dest, '|',
            CRSDepTime
        )
    ) AS unique_flights

FROM weather_ml_export
""").df()

,rows,unique_flights
0,6879484,6879484


In [163]:
WEATHER_ML_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "weather"
    / "flight_weather_features_2025.parquet"
)

con.sql(f"""
COPY (
    SELECT *
    FROM weather_ml_export
)
TO '{WEATHER_ML_PATH}'
(
    FORMAT PARQUET,
    COMPRESSION ZSTD
)
""")

print(f"Saved: {WEATHER_ML_PATH}")

print(
    f"Size: "
    f"{WEATHER_ML_PATH.stat().st_size / 1024**2:.2f} MB"
)

Saved: /Users/tseringgurung/Desktop/flight-operations-intelligence/data/processed/weather/flight_weather_features_2025.parquet
Size: 24.09 MB


In [164]:
con.sql(f"""
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT Reporting_Airline) AS airlines,
    COUNT(DISTINCT Origin) AS origins,

    ROUND(
        100.0 * COUNT(wx_temperature_c) / COUNT(*),
        2
    ) AS temperature_coverage_pct,

    ROUND(
        100.0 * COUNT(wx_wind_speed_ms) / COUNT(*),
        2
    ) AS wind_coverage_pct

FROM read_parquet('{WEATHER_ML_PATH}')
""").df()

,rows,airlines,origins,temperature_coverage_pct,wind_coverage_pct
0,6879484,14,352,98.39,98.29
